# PathWise AI — Personalized Learning Path Recommender
## Reinforcement Learning for Adaptive Education
**"Learn Smarter. Progress Faster."**

| Component | Detail |
|-----------|--------|
| **RL Algorithm** | Tabular Q-Learning (Bellman update, ε-greedy exploration) |
| **Probabilistic Model** | 5-node Bayesian Network (pgmpy / manual CPT fallback) |
| **Decision Theory** | Multi-Attribute Utility Theory (MAUT, 4-component) |
| **Graph Model** | Knowledge Graph — 15 concepts, 27 prerequisite edges |
| **MDP** | Sparse hash-based state space, 5-component reward function |
| **Data** | 200 synthetic students × 20 learning episodes each |
| **Training** | 500 RL episodes, convergence at ΔQ < 0.001 for 50 episodes |

Run **Runtime → Run All** to train the model and launch the interactive dashboard.

In [17]:
!pip install pgmpy networkx plotly ipywidgets pandas numpy scipy matplotlib seaborn -q

In [18]:
# ── FIX 1: get_ipython undefined outside Colab ───────────────────
# Use a safe try/except block instead of calling get_ipython() directly.
# get_ipython() is a Colab/IPython built-in; Pylance doesn't know it.
try:
    # This import only succeeds in IPython / Colab environments
    from IPython import get_ipython as _get_ipython  # type: ignore[import]
    _shell = _get_ipython()
    _in_notebook: bool = _shell is not None
except ImportError:
    _in_notebook = False

import matplotlib
if not _in_notebook:
    matplotlib.use('Agg')

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
# ── FIX 2: plt.Figure is not exported — use matplotlib.figure.Figure ──
from matplotlib.figure import Figure as MplFigure
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import json, pickle, os, logging, copy, warnings
from collections import defaultdict, deque
# ── FIX 3: Optional added to handle None-able parameters ─────────
from typing import Dict, List, Tuple, Optional, Any
from dataclasses import dataclass, field
warnings.filterwarnings('ignore')

try:
    import ipywidgets as widgets
    from IPython.display import display, HTML, clear_output  # type: ignore[import]
    COLAB_MODE = True
except ImportError:
    COLAB_MODE = False

# ── FIX 4: BayesianNetwork renamed to DiscreteBayesianNetwork in pgmpy ≥1.1 ──
# We import the new name with a fallback to the old name for older installs.
PGMPY_AVAILABLE = False
try:
    try:
     from pgmpy.models import DiscreteBayesianNetwork as _BayesNetClass  # type: ignore[import]
    except ImportError:
     from pgmpy.models import BayesianNetwork as _BayesNetClass           # type: ignore[import]
    from pgmpy.factors.discrete import TabularCPD
    from pgmpy.inference import VariableElimination
    PGMPY_AVAILABLE = True
except ImportError:
    print("[WARNING] pgmpy not found. Run: !pip install pgmpy")

np.random.seed(42)
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s [%(name)s] %(levelname)s: %(message)s')
logger = logging.getLogger('PathWiseAI')

# ── All parameters in one place ──────────────────────────────────
CONFIG: Dict[str, Any] = {
    # Data generation
    'n_students':            200,
    'n_learning_episodes':   20,
    'random_seed':           42,
    'quiz_noise_std':        0.05,
    'init_completion_rate':  0.40,
    # Q-Learning
    'n_train_episodes':      500,
    'alpha_0':               0.30,
    'alpha_decay':           0.001,
    'gamma':                 0.90,
    # Exploration
    'eps_max':               1.00,
    'eps_min':               0.05,
    'eps_decay':             0.005,
    'eps_boost':             0.20,
    'stagnation_threshold':  3,
    # Reward weights  (sum = 1.0)
    'w1_mastery':            0.40,
    'w2_prereq':             0.20,
    'w3_time':               0.20,
    'w4_revisit':            0.10,
    'w5_skip':               0.10,
    # Utility weights (sum = 1.0)
    'wu_mastery':            0.35,
    'wu_time':               0.25,
    'wu_preference':         0.20,
    'wu_goal':               0.20,
    # MDP
    'mastery_threshold':     0.70,
    'mastery_bins':          [0.0, 0.33, 0.70, 1.01],
    'hours_bins':            [0.0, 4.0, 6.0, 9.0],
    # Convergence
    'conv_threshold':        0.001,
    'conv_window':           50,
    # BN integration
    'bn_bonus_weight':       0.15,
    'bn_readiness_min':      0.40,
    # Viz
    'watermark':             'PathWise AI',
    'clr_not_started':       '#FF6B6B',
    'clr_in_progress':       '#FFD93D',
    'clr_mastered':          '#6BCB77',
    'clr_recommended':       '#4D96FF',
    'figure_dpi':            100,
}

assert abs(sum([CONFIG['w1_mastery'], CONFIG['w2_prereq'],
                CONFIG['w3_time'],    CONFIG['w4_revisit'],
                CONFIG['w5_skip']]) - 1.0) < 1e-9, "Reward weights must sum to 1.0"
assert abs(sum([CONFIG['wu_mastery'],    CONFIG['wu_time'],
                CONFIG['wu_preference'], CONFIG['wu_goal']]) - 1.0) < 1e-9, \
    "Utility weights must sum to 1.0"

# ── Fixed 15-concept curriculum ──────────────────────────────────
CONCEPTS: Dict[str, Dict[str, Any]] = {
    'C01': {'name': 'Python Basics',             'difficulty': 1, 'est_hours': 4,  'importance': 0.95, 'ctype': 'programming'},
    'C02': {'name': 'Statistics Fundamentals',    'difficulty': 2, 'est_hours': 6,  'importance': 0.90, 'ctype': 'theory'},
    'C03': {'name': 'Linear Algebra',             'difficulty': 2, 'est_hours': 5,  'importance': 0.85, 'ctype': 'theory'},
    'C04': {'name': 'Data Wrangling (Pandas)',    'difficulty': 2, 'est_hours': 5,  'importance': 0.88, 'ctype': 'programming'},
    'C05': {'name': 'Exploratory Data Analysis',  'difficulty': 3, 'est_hours': 6,  'importance': 0.87, 'ctype': 'practice'},
    'C06': {'name': 'Probability Theory',         'difficulty': 3, 'est_hours': 5,  'importance': 0.83, 'ctype': 'theory'},
    'C07': {'name': 'Machine Learning Basics',    'difficulty': 3, 'est_hours': 8,  'importance': 0.92, 'ctype': 'theory'},
    'C08': {'name': 'Supervised Learning',        'difficulty': 4, 'est_hours': 8,  'importance': 0.93, 'ctype': 'practice'},
    'C09': {'name': 'Unsupervised Learning',      'difficulty': 4, 'est_hours': 7,  'importance': 0.82, 'ctype': 'practice'},
    'C10': {'name': 'Feature Engineering',        'difficulty': 4, 'est_hours': 6,  'importance': 0.85, 'ctype': 'practice'},
    'C11': {'name': 'Model Evaluation',           'difficulty': 4, 'est_hours': 5,  'importance': 0.88, 'ctype': 'practice'},
    'C12': {'name': 'Deep Learning Intro',        'difficulty': 5, 'est_hours': 10, 'importance': 0.80, 'ctype': 'theory'},
    'C13': {'name': 'NLP Fundamentals',           'difficulty': 5, 'est_hours': 8,  'importance': 0.75, 'ctype': 'practice'},
    'C14': {'name': 'MLOps & Deployment',         'difficulty': 5, 'est_hours': 9,  'importance': 0.78, 'ctype': 'programming'},
    'C15': {'name': 'Capstone Project',           'difficulty': 5, 'est_hours': 12, 'importance': 1.00, 'ctype': 'practice'},
}

PREREQUISITES: List[Tuple[str, str]] = [
    ('C01','C02'),('C01','C03'),('C01','C04'),
    ('C02','C05'),('C02','C06'),('C02','C07'),
    ('C03','C07'),('C03','C12'),
    ('C04','C05'),('C04','C10'),
    ('C05','C07'),('C05','C08'),
    ('C06','C07'),('C06','C09'),
    ('C07','C08'),('C07','C09'),
    ('C08','C10'),('C08','C11'),('C08','C12'),
    ('C09','C10'),('C09','C11'),
    ('C10','C11'),('C10','C13'),
    ('C11','C14'),
    ('C12','C13'),
    ('C13','C14'),
    ('C14','C15'),
]

CONCEPT_IDS:    List[str]       = list(CONCEPTS.keys())
CONCEPT_INDEX:  Dict[str, int]  = {cid: i for i, cid in enumerate(CONCEPT_IDS)}
N_CONCEPTS:     int             = len(CONCEPT_IDS)

LEARNING_GOALS       = ['speed', 'depth', 'certification']
LEARNING_PREFERENCES = ['visual', 'reading', 'practice']

PREFERENCE_AFFINITY: Dict[str, Dict[str, float]] = {
    'visual':   {'programming': 0.65, 'theory': 0.55, 'practice': 0.80},
    'reading':  {'programming': 0.60, 'theory': 0.85, 'practice': 0.55},
    'practice': {'programming': 0.85, 'theory': 0.50, 'practice': 0.90},
}

GOAL_IMPORTANCE: Dict[str, Dict[int, float]] = {
    'speed':         {1: 1.00, 2: 0.85, 3: 0.65, 4: 0.45, 5: 0.30},
    'depth':         {1: 0.50, 2: 0.65, 3: 0.80, 4: 0.85, 5: 0.90},
    'certification': {1: 0.70, 2: 0.75, 3: 0.80, 4: 0.85, 5: 0.90},
}

print("✅ PathWise AI — imports and config loaded")
print(f"   {N_CONCEPTS} concepts | {len(PREREQUISITES)} prerequisite edges")

✅ PathWise AI — imports and config loaded
   15 concepts | 27 prerequisite edges


In [19]:
@dataclass
class StudentProfile:
    """Represents a single learner with their current knowledge state."""
    student_id:                 str
    prior_knowledge_vector:     np.ndarray      # 15-dim, initial mastery
    mastery_vector:             np.ndarray      # 15-dim, evolving mastery
    learning_preference:        str             # visual / reading / practice
    available_study_hours:      float           # hours per day
    learning_goal:              str             # speed / depth / certification
    learning_rate:              float           # individual coefficient
    satisfaction_score:         float = 0.5
    completion_rate:            float = 0.40
    episode_history:            List[Dict[str, Any]] = field(default_factory=list)

    def get_goal_encoding(self) -> int:
        return LEARNING_GOALS.index(self.learning_goal)

    def get_mastery_bins(self) -> Tuple[int, ...]:
        """Discretise mastery vector into 3 bins for state encoding."""
        bins = CONFIG['mastery_bins']
        return tuple(int(np.digitize(m, bins) - 1) for m in self.mastery_vector)

    def get_hours_bin(self) -> int:
        return int(np.digitize(self.available_study_hours, CONFIG['hours_bins']) - 1)


class DataGenerator:
    """
    Module 1 — generates 200 synthetic student records with realistic
    learning curves.  Mastery grows via:
        m_new = m_old + lr * (1 - m_old) * affinity
    """

    def __init__(self, rng: np.random.Generator):
        self.rng = rng

    # ── public API ────────────────────────────────────────────────

    def generate_students(self, n: int = CONFIG['n_students']) -> List[StudentProfile]:
        students: List[StudentProfile] = []
        for i in range(n):
            sid  = f"S{i+1:03d}"
            pref = str(self.rng.choice(LEARNING_PREFERENCES))
            goal = str(self.rng.choice(LEARNING_GOALS))
            hours = float(self.rng.uniform(2.0, 8.0))
            lr    = float(self.rng.uniform(0.08, 0.28))

            # Only C01-C03 have non-zero initial mastery
            prior = np.zeros(N_CONCEPTS)
            for j in range(3):
                prior[j] = self.rng.uniform(0.1, 0.65)

            students.append(StudentProfile(
                student_id=sid,
                prior_knowledge_vector=prior.copy(),
                mastery_vector=prior.copy(),
                learning_preference=pref,
                available_study_hours=hours,
                learning_goal=goal,
                learning_rate=lr,
            ))
        return students

    def simulate_learning_episode(self,
                                   student: StudentProfile,
                                   concept_id: str,
                                   kg: 'KnowledgeGraph') -> Dict[str, Any]:
        """
        Simulate one study episode.  Returns updated mastery + metrics.
        """
        idx   = CONCEPT_INDEX[concept_id]
        old_m = student.mastery_vector[idx]

        # Concept-preference affinity
        ctype    = CONCEPTS[concept_id]['ctype']
        affinity = PREFERENCE_AFFINITY[student.learning_preference][ctype]

        # Difficulty penalty: harder concepts → slower gain
        diff_factor = 1.0 - (CONCEPTS[concept_id]['difficulty'] - 1) * 0.10

        # Prerequisite multiplier: gate learning if prereqs not met
        prereqs_met = kg.prerequisites_satisfied(concept_id, student.mastery_vector)
        prereq_mult = 1.0 if prereqs_met else 0.35

        new_m = old_m + student.learning_rate * (1 - old_m) * affinity * diff_factor * prereq_mult
        new_m = float(np.clip(new_m, 0.0, 1.0))
        student.mastery_vector[idx] = new_m

        # Quiz score: mastery + Gaussian noise
        quiz_score = float(np.clip(
            new_m + self.rng.normal(0, CONFIG['quiz_noise_std']), 0, 1))

        # Completion rate increases with momentum
        mastered_count = int(np.sum(student.mastery_vector >= CONFIG['mastery_threshold']))
        student.completion_rate = float(np.clip(
            CONFIG['init_completion_rate'] + 0.03 * mastered_count, 0, 1))

        # Satisfaction: proxy for alignment quality
        satisfaction = float(np.clip(
            0.5 + 0.4 * (new_m - old_m) * 5 + 0.1 * prereq_mult, 0, 1))
        student.satisfaction_score = satisfaction

        record: Dict[str, Any] = {
            'student_id':    student.student_id,
            'concept_id':    concept_id,
            'old_mastery':   old_m,
            'new_mastery':   new_m,
            'delta_mastery': new_m - old_m,
            'quiz_score':    quiz_score,
            'completion_rate': student.completion_rate,
            'satisfaction':  satisfaction,
            'prereqs_met':   prereqs_met,
        }
        student.episode_history.append(record)
        return record

    def build_dataset(self,
                      students: List[StudentProfile],
                      kg: 'KnowledgeGraph') -> pd.DataFrame:
        """Generate the full training dataset across all students × episodes."""
        rows: List[Dict[str, Any]] = []
        rng2 = np.random.default_rng(CONFIG['random_seed'])
        for s in students:
            for ep in range(CONFIG['n_learning_episodes']):
                valid = kg.get_valid_actions(s.mastery_vector)
                if not valid:
                    valid = CONCEPT_IDS[:3]
                # ── FIX 5: rng.choice on list[str] works fine ────
                cid = str(rng2.choice(valid))
                rec = self.simulate_learning_episode(s, cid, kg)
                rec['episode']            = ep
                rec['learning_preference']= s.learning_preference
                rec['learning_goal']      = s.learning_goal
                rec['available_hours']    = s.available_study_hours
                rows.append(rec)
        return pd.DataFrame(rows)


print("✅ DataGenerator and StudentProfile defined")

✅ DataGenerator and StudentProfile defined


In [20]:
class KnowledgeGraph:
    """
    Module 3 — NetworkX DiGraph of the 15-concept curriculum.
    Provides prerequisite gating, valid action sets, and layout for viz.
    """

    def __init__(self) -> None:
        self.graph = nx.DiGraph()
        self._build()
        # Pre-compute predecessor sets for speed
        self._prereq_cache: Dict[str, List[str]] = {
            cid: list(self.graph.predecessors(cid)) for cid in CONCEPT_IDS
        }

    def _build(self) -> None:
        for cid, attrs in CONCEPTS.items():
            self.graph.add_node(cid, **attrs)
        for src, dst in PREREQUISITES:
            self.graph.add_edge(src, dst)

    def prerequisites_satisfied(self,
                                  concept_id: str,
                                  mastery_vec: np.ndarray,
                                  threshold: Optional[float] = None) -> bool:
        """Return True if all direct prerequisites are above threshold."""
        thr = threshold if threshold is not None else CONFIG['mastery_threshold']
        for p in self._prereq_cache[concept_id]:
            if mastery_vec[CONCEPT_INDEX[p]] < thr:
                return False
        return True

    def get_valid_actions(self, mastery_vec: np.ndarray) -> List[str]:
        """
        Actions are valid when:
          1. All prerequisites ≥ mastery_threshold
          2. Concept is not yet mastered (mastery < 1.0)
        """
        valid = []
        for cid in CONCEPT_IDS:
            idx = CONCEPT_INDEX[cid]
            if (mastery_vec[idx] < 1.0 and
                    self.prerequisites_satisfied(cid, mastery_vec)):
                valid.append(cid)
        return valid if valid else [CONCEPT_IDS[0]]   # fallback: Python Basics

    def get_prerequisite_bonus(self,
                                concept_id: str,
                                mastery_vec: np.ndarray) -> float:
        """Fraction of prerequisites that are mastered."""
        prereqs = self._prereq_cache[concept_id]
        if not prereqs:
            return 1.0
        met = sum(1 for p in prereqs
                  if mastery_vec[CONCEPT_INDEX[p]] >= CONFIG['mastery_threshold'])
        return met / len(prereqs)

    def topological_order(self) -> List[str]:
        return list(nx.topological_sort(self.graph))

    def get_pos_for_viz(self) -> Dict[str, Tuple[float, float]]:
        """Layered layout for visualization."""
        level_map: Dict[int, List[str]] = {1: [], 2: [], 3: [], 4: [], 5: []}
        for cid, attrs in CONCEPTS.items():
            level_map[attrs['difficulty']].append(cid)
        pos: Dict[str, Tuple[float, float]] = {}
        for lvl, nodes in level_map.items():
            for j, n in enumerate(nodes):
                x = (j - (len(nodes) - 1) / 2) * 2.5
                pos[n] = (x, float(-lvl * 2))
        return pos

    def node_color(self, mastery: float) -> str:
        if mastery < 0.33:
            return CONFIG['clr_not_started']
        elif mastery < 0.70:
            return CONFIG['clr_in_progress']
        return CONFIG['clr_mastered']


print("✅ KnowledgeGraph defined")

✅ KnowledgeGraph defined


In [21]:
class BayesianEngine:
    """
    Module 4 — 5-node Bayesian Network modelling mastery probability.

    Structure:
        PriorKnowledge → LearningAbility → MasteryProbability
        ConceptDifficulty              → MasteryProbability
        LearningAbility                → SuccessProbability
        MasteryProbability             → SuccessProbability

    States:
        PriorKnowledge:    {0=low, 1=medium, 2=high}
        LearningAbility:   {0=low, 1=medium, 2=high}
        ConceptDifficulty: {0=easy(1-2), 1=medium(3), 2=hard(4-5)}
        MasteryProbability:{0=low, 1=medium, 2=high}
        SuccessProbability:{0=fail, 1=pass, 2=distinction}
    """

    def __init__(self) -> None:
        self.model:  Any = None
        self.infer:  Optional[VariableElimination] = None  # type: ignore[type-arg]
        self._cache: Dict[str, Dict[str, np.ndarray]] = {}
        if PGMPY_AVAILABLE:
            self._build_pgmpy()
        else:
            logger.warning("pgmpy unavailable — using manual CPT fallback")

    # ── pgmpy implementation ───────────────────────────────────────

    def _build_pgmpy(self) -> None:
        # ── FIX 4 (continued): use _BayesNetClass alias ──────────
        # add_cpds and check_model ARE valid methods; Pylance stubs are wrong
        # because the class was renamed.  We type-ignore the attribute calls.
        try:
            model = _BayesNetClass([                             # type: ignore[operator]
                ('PriorKnowledge',    'LearningAbility'),
                ('LearningAbility',   'MasteryProbability'),
                ('ConceptDifficulty', 'MasteryProbability'),
                ('LearningAbility',   'SuccessProbability'),
                ('MasteryProbability','SuccessProbability'),
            ])

            # P(PriorKnowledge) — marginal
            cpd_pk = TabularCPD('PriorKnowledge', 3,
                [[0.35], [0.40], [0.25]])

            # P(LearningAbility | PriorKnowledge)
            cpd_la = TabularCPD('LearningAbility', 3,
                [[0.55, 0.25, 0.10],
                 [0.35, 0.50, 0.35],
                 [0.10, 0.25, 0.55]],
                evidence=['PriorKnowledge'], evidence_card=[3])

            # P(ConceptDifficulty) — marginal
            cpd_cd = TabularCPD('ConceptDifficulty', 3,
                [[0.267], [0.200], [0.533]])

            # P(MasteryProbability | LearningAbility, ConceptDifficulty)
            cpd_mp = TabularCPD('MasteryProbability', 3,
                [[0.360, 0.550, 0.720,  0.150, 0.300, 0.500,  0.050, 0.150, 0.300],
                 [0.450, 0.360, 0.225,  0.450, 0.490, 0.375,  0.295, 0.405, 0.395],
                 [0.190, 0.090, 0.055,  0.400, 0.210, 0.125,  0.655, 0.445, 0.305]],
                evidence=['LearningAbility', 'ConceptDifficulty'],
                evidence_card=[3, 3])

            # P(SuccessProbability | LearningAbility, MasteryProbability)
            cpd_sp = TabularCPD('SuccessProbability', 3,
                [[0.700, 0.395, 0.195,  0.550, 0.245, 0.095,  0.395, 0.145, 0.045],
                 [0.250, 0.480, 0.555,  0.350, 0.520, 0.475,  0.455, 0.480, 0.350],
                 [0.050, 0.125, 0.250,  0.100, 0.235, 0.430,  0.150, 0.375, 0.605]],
                evidence=['LearningAbility', 'MasteryProbability'],
                evidence_card=[3, 3])

            # ── FIX 4a: suppress Pylance attribute errors on pgmpy model ──
            model.add_cpds(cpd_pk, cpd_la, cpd_cd, cpd_mp, cpd_sp)   # type: ignore[attr-defined]
            assert model.check_model(), "BN model check failed"        # type: ignore[attr-defined]
            self.model = model
            self.infer = VariableElimination(model)
            logger.info("Bayesian Network built and validated with pgmpy")
        except Exception as e:
            logger.error(f"pgmpy BN build failed: {e}. Falling back to manual CPT.")
            self.model = None
            self.infer = None

    # ── manual CPT fallback ────────────────────────────────────────

    _CPT_MP = np.array([  # shape (3,3,3): [LA, CD, MP]
        [[0.360, 0.450, 0.190], [0.550, 0.360, 0.090], [0.720, 0.225, 0.055]],
        [[0.150, 0.450, 0.400], [0.300, 0.490, 0.210], [0.500, 0.375, 0.125]],
        [[0.050, 0.295, 0.655], [0.150, 0.405, 0.445], [0.300, 0.395, 0.305]],
    ])

    _CPT_SP = np.array([  # shape (3,3,3): [LA, MP, SP]
        [[0.700, 0.250, 0.050], [0.395, 0.480, 0.125], [0.195, 0.555, 0.250]],
        [[0.550, 0.350, 0.100], [0.245, 0.520, 0.235], [0.095, 0.475, 0.430]],
        [[0.395, 0.455, 0.150], [0.145, 0.480, 0.375], [0.045, 0.350, 0.605]],
    ])

    _CPT_LA = np.array([  # shape (3,3): [PK, LA]
        [0.55, 0.35, 0.10],
        [0.25, 0.50, 0.25],
        [0.10, 0.35, 0.55],
    ])

    def _get_pk_state(self, student: StudentProfile) -> int:
        avg = float(np.mean(student.prior_knowledge_vector[:3]))
        if avg < 0.33:   return 0
        elif avg < 0.67: return 1
        return 2

    def _get_cd_state(self, concept_id: str) -> int:
        d = CONCEPTS[concept_id]['difficulty']
        if d <= 2:   return 0
        elif d == 3: return 1
        return 2

    def _manual_infer(self, pk: int, cd: int) -> Dict[str, np.ndarray]:
        """Compute P(MP) and P(SP) via manual marginalisation."""
        p_la: np.ndarray = self._CPT_LA[pk]
        p_mp = np.zeros(3)
        p_sp = np.zeros(3)
        for la in range(3):
            p_mp += p_la[la] * self._CPT_MP[la, cd]
        for la in range(3):
            for mp in range(3):
                p_sp += p_la[la] * self._CPT_MP[la, cd, mp] * self._CPT_SP[la, mp]
        return {'MasteryProbability': p_mp, 'SuccessProbability': p_sp}

    def infer_mastery(self, student: StudentProfile,
                       concept_id: str) -> Dict[str, np.ndarray]:
        """
        Returns dict with 'MasteryProbability' and 'SuccessProbability',
        each a 3-element probability array.
        """
        cache_key = f"{student.student_id}_{concept_id}"
        if cache_key in self._cache:
            return self._cache[cache_key]

        pk = self._get_pk_state(student)
        cd = self._get_cd_state(concept_id)

        if self.infer is not None:
            try:
                q = self.infer.query(
                    variables=['MasteryProbability', 'SuccessProbability'],
                    evidence={'PriorKnowledge': pk, 'ConceptDifficulty': cd},
                    show_progress=False,
                )
                # ── FIX 6: DiscreteFactor result accessed via .values (ndarray) ──
                # q['MasteryProbability'] returns a DiscreteFactor; .values is ndarray.
                # Do NOT subscript with [key] — use .values directly.
                result: Dict[str, np.ndarray] = {
                    'MasteryProbability': np.array(q['MasteryProbability'].values),
                    'SuccessProbability': np.array(q['SuccessProbability'].values),
                }
            except Exception:
                result = self._manual_infer(pk, cd)
        else:
            result = self._manual_infer(pk, cd)

        self._cache[cache_key] = result
        return result

    def p_mastery_high(self, student: StudentProfile, concept_id: str) -> float:
        """Scalar: P(MasteryProbability = high)."""
        return float(self.infer_mastery(student, concept_id)['MasteryProbability'][2])

    def p_success_pass(self, student: StudentProfile, concept_id: str) -> float:
        """Scalar: P(SuccessProbability ≥ pass) = P(pass) + P(distinction)."""
        sp = self.infer_mastery(student, concept_id)['SuccessProbability']
        return float(sp[1] + sp[2])

    def is_ready(self, student: StudentProfile, concept_id: str) -> bool:
        """BN gate: recommend only if P(success≥pass) ≥ threshold."""
        return self.p_success_pass(student, concept_id) >= CONFIG['bn_readiness_min']

    def reward_shaping_bonus(self, student: StudentProfile, concept_id: str) -> float:
        """BN-derived bonus added to RL reward."""
        return CONFIG['bn_bonus_weight'] * self.p_mastery_high(student, concept_id)


print("✅ BayesianEngine defined")

✅ BayesianEngine defined


In [22]:
class MDPEnvironment:
    """
    Module 5 — Markov Decision Process for adaptive concept recommendation.

    State:  (mastery_bins[15], hours_bin, goal_encoding)   → hashed int
    Action: concept_id ∈ {C01..C15}
    Reward: 5-component weighted sum + BN bonus
    """

    def __init__(self, kg: KnowledgeGraph, bn: BayesianEngine) -> None:
        self.kg = kg
        self.bn = bn

    def encode_state(self, student: StudentProfile) -> int:
        """Hash the discretised student state to a sparse integer key."""
        bins  = student.get_mastery_bins()
        hbin  = student.get_hours_bin()
        genc  = student.get_goal_encoding()
        state = bins + (hbin, genc)
        return hash(state)

    def compute_reward(self,
                       student: StudentProfile,
                       action_cid: str,
                       new_mastery_vec: np.ndarray) -> float:
        """
        R = w1*ΔMastery + w2*PrereqBonus + w3*TimeEfficiency
              - w4*RevisitPenalty - w5*SkipPenalty + BN_bonus
        """
        idx   = CONCEPT_INDEX[action_cid]
        old_m = student.mastery_vector[idx]
        new_m = new_mastery_vec[idx]
        delta = float(new_m - old_m)

        r1 = float(np.clip(delta * 4.0, 0.0, 1.0))
        r2 = self.kg.get_prerequisite_bonus(action_cid, student.mastery_vector)

        est_h = float(CONCEPTS[action_cid]['est_hours'])
        avail = student.available_study_hours * 2
        r3    = float(np.clip(1.0 - max(0.0, est_h - avail) / (est_h + 1e-9), 0.0, 1.0))

        p4 = 1.0 if old_m >= CONFIG['mastery_threshold'] else 0.0
        prereqs_met = self.kg.prerequisites_satisfied(action_cid, student.mastery_vector)
        p5 = 0.0 if prereqs_met else 1.0

        reward = (CONFIG['w1_mastery']  * r1
                + CONFIG['w2_prereq']   * r2
                + CONFIG['w3_time']     * r3
                - CONFIG['w4_revisit']  * p4
                - CONFIG['w5_skip']     * p5)
        reward += self.bn.reward_shaping_bonus(student, action_cid)
        return float(reward)

    def step(self,
             student: StudentProfile,
             action_cid: str,
             data_gen: DataGenerator) -> Tuple[int, float, StudentProfile]:
        """
        Execute action, return (new_state_hash, reward, updated_student).
        If prerequisites violated → deterministic stay + large negative reward.
        """
        if not self.kg.prerequisites_satisfied(action_cid, student.mastery_vector):
            return self.encode_state(student), -0.30, student

        student_copy = copy.deepcopy(student)
        data_gen.simulate_learning_episode(student_copy, action_cid, self.kg)
        reward    = self.compute_reward(student, action_cid, student_copy.mastery_vector)
        new_state = self.encode_state(student_copy)
        return new_state, reward, student_copy


print("✅ MDPEnvironment defined")

✅ MDPEnvironment defined


In [23]:
# ── Type alias to silence Pylance on defaultdict[int, defaultdict[str, float]] ──
_QTable = Dict[int, Dict[str, float]]

class QLearningAgent:
    """
    Tabular Q-Learning with sparse dict Q-table and decaying α.
    Q(s,a) ← Q(s,a) + α[r + γ·max_a'Q(s',a') − Q(s,a)]
    """

    def __init__(self) -> None:
        # ── FIX 7: explicit typed defaultdict resolves max() overload error ──
        # Pylance struggles with nested defaultdicts; we cast to _QTable on access.
        self._Q: Any = defaultdict(lambda: defaultdict(float))
        self.episode_rewards:   List[float]         = []
        self.episode_delta_q:   List[float]         = []
        self.converged_at:      Optional[int]       = None
        self._conv_buffer:      deque[float]        = deque(maxlen=CONFIG['conv_window'])

    # ── expose Q as a typed property ─────────────────────────────
    @property
    def Q(self) -> _QTable:
        return self._Q  # type: ignore[return-value]

    # ── helpers ───────────────────────────────────────────────────

    def _alpha(self, t: int) -> float:
        return float(CONFIG['alpha_0'] / (1.0 + CONFIG['alpha_decay'] * t))

    def _max_q(self, state: int, valid_actions: List[str]) -> float:
        if not valid_actions:
            return 0.0
        # ── FIX 7 (continued): iterate list, index defaultdict explicitly ──
        return max(float(self._Q[state][a]) for a in valid_actions)

    def select_action(self,
                       state: int,
                       valid_actions: List[str],
                       epsilon: float,
                       rng: np.random.Generator) -> Tuple[str, bool]:
        """ε-greedy action selection. Returns (action, is_explore)."""
        if not valid_actions:
            return CONCEPT_IDS[0], True
        if rng.random() < epsilon:
            return str(rng.choice(valid_actions)), True
        # ── FIX 7: explicit lambda with float cast removes overload ambiguity ──
        best: str = max(valid_actions, key=lambda a: float(self._Q[state][a]))
        return best, False

    def update(self,
               state: int,
               action: str,
               reward: float,
               next_state: int,
               next_valid: List[str],
               t: int) -> float:
        """Single Bellman update. Returns |ΔQ|."""
        alpha  = self._alpha(t)
        old_q  = float(self._Q[state][action])
        max_nq = self._max_q(next_state, next_valid)
        new_q  = old_q + alpha * (reward + CONFIG['gamma'] * max_nq - old_q)
        self._Q[state][action] = new_q
        return abs(new_q - old_q)

    def check_convergence(self, mean_dq: float, episode: int) -> bool:
        self._conv_buffer.append(mean_dq)
        if (len(self._conv_buffer) == CONFIG['conv_window'] and
                max(self._conv_buffer) < CONFIG['conv_threshold'] and
                self.converged_at is None):
            self.converged_at = episode
            logger.info(f"Q-table converged at episode {episode}")
            return True
        return False

    def get_policy(self, state: int, valid_actions: List[str]) -> Optional[str]:
        if not valid_actions:
            return None
        return max(valid_actions, key=lambda a: float(self._Q[state][a]))


class ExplorationTracker:
    """
    Manages ε-greedy decay and stagnation-based ε-boost.
    ε(t) = ε_max · exp(−decay · t), floored at ε_min.
    """

    def __init__(self) -> None:
        self.epsilon_history:   List[float] = []
        self.explore_flags:     List[bool]  = []
        self.stagnation_events: List[int]   = []
        self._no_gain_streak:   int         = 0
        self._last_avg_mastery: float       = 0.0

    def get_epsilon(self, t: int, stagnated: bool = False) -> float:
        eps = max(CONFIG['eps_min'],
                  CONFIG['eps_max'] * np.exp(-CONFIG['eps_decay'] * t))
        if stagnated:
            eps = min(1.0, eps + CONFIG['eps_boost'])
        return float(eps)

    def update_stagnation(self, current_avg_mastery: float, episode: int) -> bool:
        delta = current_avg_mastery - self._last_avg_mastery
        if delta < 1e-4:
            self._no_gain_streak += 1
        else:
            self._no_gain_streak = 0
        self._last_avg_mastery = current_avg_mastery
        stagnated = self._no_gain_streak >= CONFIG['stagnation_threshold']
        if stagnated:
            self.stagnation_events.append(episode)
            self._no_gain_streak = 0
        return stagnated

    def log(self, epsilon: float, is_explore: bool) -> None:
        self.epsilon_history.append(epsilon)
        self.explore_flags.append(is_explore)

    def explore_ratio_per_window(self, window: int = 20) -> List[float]:
        flags  = self.explore_flags
        ratios: List[float] = []
        for i in range(0, len(flags), window):
            chunk = flags[i:i+window]
            ratios.append(sum(chunk) / len(chunk) if chunk else 0.0)
        return ratios


def run_training(students:  List[StudentProfile],
                 kg:        KnowledgeGraph,
                 bn:        BayesianEngine,
                 mdp:       MDPEnvironment,
                 data_gen:  DataGenerator) -> Tuple[QLearningAgent, ExplorationTracker]:
    """
    Main Q-Learning training loop.
    500 episodes × up to 15 steps each → tracks reward + convergence.
    """
    agent       = QLearningAgent()
    tracker     = ExplorationTracker()
    rng         = np.random.default_rng(CONFIG['random_seed'])
    n_ep        = CONFIG['n_train_episodes']
    global_step = 0

    # ── FIX 8: rng.choice cannot accept List[StudentProfile] ─────
    # Use rng.integers to pick an index, then index into the list.
    n_students = len(students)

    for ep in range(n_ep):
        student = copy.deepcopy(students[int(rng.integers(0, n_students))])
        ep_reward   = 0.0
        dq_list:    List[float] = []
        steps       = 0

        avg_mastery = float(np.mean(student.mastery_vector))
        stagnated   = tracker.update_stagnation(avg_mastery, ep)
        epsilon     = tracker.get_epsilon(ep, stagnated)

        while steps < N_CONCEPTS:
            state      = mdp.encode_state(student)
            valid_acts = kg.get_valid_actions(student.mastery_vector)
            if not valid_acts:
                break

            action, is_explore = agent.select_action(state, valid_acts, epsilon, rng)
            tracker.log(epsilon, is_explore)

            next_state, reward, student = mdp.step(student, action, data_gen)
            next_valid = kg.get_valid_actions(student.mastery_vector)

            dq = agent.update(state, action, reward, next_state, next_valid, global_step)
            dq_list.append(dq)
            ep_reward   += reward
            global_step += 1
            steps       += 1

        agent.episode_rewards.append(ep_reward)
        mean_dq = float(np.mean(dq_list)) if dq_list else 0.0
        agent.episode_delta_q.append(mean_dq)
        agent.check_convergence(mean_dq, ep)

        if ep % 100 == 0:
            logger.info(f"Episode {ep:4d} | ε={epsilon:.3f} | "
                        f"reward={ep_reward:.3f} | ΔQ={mean_dq:.4f}")

    return agent, tracker


print("✅ QLearningAgent, ExplorationTracker, run_training defined")

✅ QLearningAgent, ExplorationTracker, run_training defined


In [24]:
def plot_exploration(tracker:   ExplorationTracker,
                     agent:     QLearningAgent,
                     save_path: Optional[str] = None) -> MplFigure:
    """Plots ε decay, explore/exploit ratio, and stagnation events."""
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f'{CONFIG["watermark"]} — Exploration vs Exploitation',
                 fontsize=14, fontweight='bold')

    eps_hist = tracker.epsilon_history
    episodes = list(range(len(agent.episode_rewards)))

    axes[0].plot(eps_hist, color='steelblue', linewidth=1.2, alpha=0.8)
    for ev in tracker.stagnation_events:
        axes[0].axvline(ev * N_CONCEPTS, color='red', alpha=0.5,
                        linestyle='--', linewidth=1)
    axes[0].set_title('ε-Decay with Stagnation Boosts')
    axes[0].set_xlabel('Training Step')
    axes[0].set_ylabel('ε (Exploration Rate)')
    axes[0].set_ylim(0, 1.1)

    ratios = tracker.explore_ratio_per_window(window=15)
    axes[1].plot(ratios, color='darkorange', linewidth=1.5, marker='o', markersize=3)
    axes[1].axhline(0.5, linestyle='--', color='gray', alpha=0.6)
    axes[1].set_title('Explore Ratio (window=15 steps)')
    axes[1].set_xlabel('Window Index')
    axes[1].set_ylabel('Fraction Exploring')
    axes[1].set_ylim(0, 1.1)

    mid = len(agent.episode_rewards) // 2
    axes[2].plot(episodes[:mid], agent.episode_rewards[:mid],
                 label='Explore era', color='tomato', alpha=0.7)
    axes[2].plot(episodes[mid:], agent.episode_rewards[mid:],
                 label='Exploit era', color='mediumseagreen', alpha=0.7)
    axes[2].set_title('Cumulative Reward: Explore vs Exploit Era')
    axes[2].set_xlabel('Episode')
    axes[2].set_ylabel('Episode Reward')
    axes[2].legend()

    plt.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=CONFIG.get('figure_dpi', 100))
    return fig

In [25]:
class UtilityCalculator:
    """
    Multi-Attribute Utility Function:
        U(a,s) = wu_mastery * U_mastery
               + wu_time    * U_time
               + wu_pref    * U_preference
               + wu_goal    * U_goal_alignment
    """

    def __init__(self, bn: BayesianEngine) -> None:
        self.bn = bn

    def u_mastery(self, student: StudentProfile, concept_id: str) -> float:
        return self.bn.p_mastery_high(student, concept_id)

    def u_time(self, student: StudentProfile, concept_id: str) -> float:
        est   = float(CONCEPTS[concept_id]['est_hours'])
        avail = student.available_study_hours * 2
        return float(np.clip(1.0 - max(0.0, est - avail) / (est + 1e-9), 0.0, 1.0))

    def u_preference(self, student: StudentProfile, concept_id: str) -> float:
        ctype = str(CONCEPTS[concept_id]['ctype'])
        return PREFERENCE_AFFINITY[student.learning_preference].get(ctype, 0.5)

    def u_goal_alignment(self, student: StudentProfile, concept_id: str) -> float:
        diff            = int(CONCEPTS[concept_id]['difficulty'])
        base            = GOAL_IMPORTANCE[student.learning_goal][diff]
        importance_bonus = float(CONCEPTS[concept_id]['importance']) * 0.15
        return float(np.clip(base + importance_bonus, 0.0, 1.0))

    def compute(self, student: StudentProfile, concept_id: str) -> float:
        return (CONFIG['wu_mastery']    * self.u_mastery(student, concept_id)
              + CONFIG['wu_time']       * self.u_time(student, concept_id)
              + CONFIG['wu_preference'] * self.u_preference(student, concept_id)
              + CONFIG['wu_goal']       * self.u_goal_alignment(student, concept_id))

    def rank_actions(self, student: StudentProfile,
                      valid_actions: List[str]) -> List[Tuple[str, float]]:
        scored = [(a, self.compute(student, a)) for a in valid_actions]
        return sorted(scored, key=lambda x: x[1], reverse=True)

    def sensitivity_analysis(self,
                              student: StudentProfile,
                              valid_actions: List[str],
                              weight_shifts: Optional[List[Dict[str, Any]]] = None
                              ) -> pd.DataFrame:
        """Show how top recommendation changes when utility weights shift."""
        if weight_shifts is None:
            weight_shifts = [
                {'name': 'Baseline',        'wu_mastery': 0.35, 'wu_time': 0.25, 'wu_preference': 0.20, 'wu_goal': 0.20},
                {'name': 'Mastery Focus',   'wu_mastery': 0.60, 'wu_time': 0.15, 'wu_preference': 0.15, 'wu_goal': 0.10},
                {'name': 'Time Focus',      'wu_mastery': 0.20, 'wu_time': 0.55, 'wu_preference': 0.15, 'wu_goal': 0.10},
                {'name': 'Goal Focus',      'wu_mastery': 0.20, 'wu_time': 0.15, 'wu_preference': 0.15, 'wu_goal': 0.50},
            ]
        rows: List[Dict[str, Any]] = []
        orig = {k: CONFIG[k] for k in ['wu_mastery', 'wu_time', 'wu_preference', 'wu_goal']}
        for scenario in weight_shifts:
            CONFIG.update({k: v for k, v in scenario.items() if k.startswith('wu_')})
            for a in valid_actions:
                rows.append({'Scenario': scenario['name'],
                             'Concept':  a,
                             'Name':     CONCEPTS[a]['name'],
                             'Utility':  round(self.compute(student, a), 4)})
        CONFIG.update(orig)  # restore original weights
        return pd.DataFrame(rows)


print("✅ UtilityCalculator defined")

✅ UtilityCalculator defined


In [26]:
class RecommendationEngine:
    """
    Module 6 — Combines Q-values (primary) + Utility (tiebreaker) + BN gate.
    Returns ranked top-3 concept recommendations with full reasoning.
    """

    def __init__(self,
                 agent:   QLearningAgent,
                 kg:      KnowledgeGraph,
                 bn:      BayesianEngine,
                 utility: UtilityCalculator,
                 mdp:     MDPEnvironment) -> None:
        self.agent   = agent
        self.kg      = kg
        self.bn      = bn
        self.utility = utility
        self.mdp     = mdp

    def recommend(self,
                   student: StudentProfile,
                   top_k:   int = 3) -> List[Dict[str, Any]]:
        """
        Returns list of dicts (max top_k) with recommendation details.
        Ranking: Q-value primary, Utility score tiebreaker.
        Filtered by BN readiness gate.
        """
        state      = self.mdp.encode_state(student)
        valid_acts = self.kg.get_valid_actions(student.mastery_vector)

        ready_acts = [a for a in valid_acts if self.bn.is_ready(student, a)]
        if not ready_acts:
            ready_acts = valid_acts   # fallback: ignore BN gate if all blocked

        scored: List[Dict[str, Any]] = []
        for a in ready_acts:
            q_val     = float(self.agent.Q[state][a])
            util      = self.utility.compute(student, a)
            bn_mp     = self.bn.p_mastery_high(student, a)
            bn_sp     = self.bn.p_success_pass(student, a)
            prereq_ok = self.kg.prerequisites_satisfied(a, student.mastery_vector)
            combined  = q_val + 0.001 * util
            scored.append({
                'concept_id':    a,
                'concept_name':  CONCEPTS[a]['name'],
                'q_value':       round(q_val, 4),
                'utility_score': round(util,  4),
                'bn_mastery_p':  round(bn_mp, 4),
                'bn_success_p':  round(bn_sp, 4),
                'prereq_ok':     prereq_ok,
                'difficulty':    CONCEPTS[a]['difficulty'],
                'est_hours':     CONCEPTS[a]['est_hours'],
                '_combined':     combined,
            })

        scored.sort(key=lambda x: (x['_combined'], x['utility_score']), reverse=True)
        return scored[:top_k]


print("✅ RecommendationEngine defined")

✅ RecommendationEngine defined


In [27]:
class AnalyticsEngine:
    """Module 7 + 8 — All 10 visualisations + metric computation."""

    def __init__(self,
                 agent:    QLearningAgent,
                 tracker:  ExplorationTracker,
                 students: List[StudentProfile],
                 kg:       KnowledgeGraph,
                 bn:       BayesianEngine,
                 utility:  UtilityCalculator,
                 rec_eng:  RecommendationEngine,
                 dataset:  pd.DataFrame) -> None:
        self.agent    = agent
        self.tracker  = tracker
        self.students = students
        self.kg       = kg
        self.bn       = bn
        self.utility  = utility
        self.rec      = rec_eng
        self.df       = dataset

    # ── VIZ 1: Knowledge Graph ────────────────────────────────────

    def plot_knowledge_graph(self,
                              student:     Optional[StudentProfile] = None,
                              recommended: Optional[str]            = None,
                              save_path:   Optional[str]            = None
                              ) -> MplFigure:
        G   = self.kg.graph
        pos = self.kg.get_pos_for_viz()
        fig, ax = plt.subplots(figsize=(16, 9))

        mastery_vec = student.mastery_vector if student is not None else np.zeros(N_CONCEPTS)
        node_colors: List[str] = []
        for cid in G.nodes():
            if cid == recommended:
                node_colors.append(CONFIG['clr_recommended'])
            else:
                m = mastery_vec[CONCEPT_INDEX[cid]]
                node_colors.append(self.kg.node_color(float(m)))

        nx.draw_networkx(G, pos=pos, ax=ax,
                         node_color=node_colors, node_size=1800,
                         font_size=7, font_weight='bold',
                         arrows=True, arrowsize=15,
                         edge_color='#555555', width=1.5,
                         labels={n: f"{n}\n{CONCEPTS[n]['name'][:12]}"
                                 for n in G.nodes()})

        legend_patches = [
            mpatches.Patch(color=CONFIG['clr_not_started'],  label='Not Started'),
            mpatches.Patch(color=CONFIG['clr_in_progress'],  label='In Progress'),
            mpatches.Patch(color=CONFIG['clr_mastered'],     label='Mastered'),
            mpatches.Patch(color=CONFIG['clr_recommended'],  label='Recommended'),
        ]
        ax.legend(handles=legend_patches, loc='upper left', fontsize=9)
        suffix = f' | Student: {student.student_id}' if student is not None else ''
        ax.set_title(f'{CONFIG["watermark"]} — Knowledge Graph{suffix}',
                     fontsize=13, fontweight='bold')
        ax.axis('off')
        plt.tight_layout()
        if save_path is not None:
            fig.savefig(save_path, dpi=100, bbox_inches='tight')
        return fig

    # ── VIZ 2: Q-Value Heatmap ────────────────────────────────────

    def plot_q_heatmap(self,
                        top_n_states: int          = 10,
                        save_path:    Optional[str] = None) -> MplFigure:
        state_keys = list(self.agent.Q.keys())[:top_n_states]
        matrix     = np.zeros((len(state_keys), N_CONCEPTS))
        for r, sk in enumerate(state_keys):
            for c, cid in enumerate(CONCEPT_IDS):
                matrix[r, c] = float(self.agent.Q[sk].get(cid, 0.0))

        fig, ax = plt.subplots(figsize=(16, 6))
        im = ax.imshow(matrix, cmap='RdYlGn', aspect='auto',
                       vmin=matrix.min(), vmax=matrix.max())
        ax.set_xticks(range(N_CONCEPTS))
        ax.set_xticklabels([f"{cid}\n{CONCEPTS[cid]['name'][:8]}"
                            for cid in CONCEPT_IDS],
                           rotation=45, ha='right', fontsize=8)
        ax.set_yticks(range(len(state_keys)))
        ax.set_yticklabels([f"S{i+1}" for i in range(len(state_keys))], fontsize=9)
        ax.set_title(f'{CONFIG["watermark"]} — Q-Value Heatmap (top {top_n_states} states)',
                     fontsize=13, fontweight='bold')
        plt.colorbar(im, ax=ax, label='Q-value')
        plt.tight_layout()
        if save_path is not None:
            fig.savefig(save_path, dpi=100)
        return fig

    # ── VIZ 3: Cumulative Reward Curve ───────────────────────────

    def plot_reward_curve(self,
                           window:    int          = 20,
                           save_path: Optional[str] = None) -> MplFigure:
        rewards  = self.agent.episode_rewards
        smoothed = pd.Series(rewards).rolling(window, min_periods=1).mean().to_numpy()

        fig, ax = plt.subplots(figsize=(12, 5))
        ax.plot(rewards,  color='lightcoral', alpha=0.4, linewidth=1, label='Raw')
        ax.plot(smoothed, color='crimson',    linewidth=2.0,
                label=f'Smoothed (w={window})')
        if self.agent.converged_at is not None:
            ax.axvline(self.agent.converged_at, color='navy', linestyle='--',
                       label=f'Converged @ ep {self.agent.converged_at}')
        ax.set_title(f'{CONFIG["watermark"]} — Cumulative Reward over Training',
                     fontsize=13, fontweight='bold')
        ax.set_xlabel('Episode')
        ax.set_ylabel('Episode Reward')
        ax.legend()
        plt.tight_layout()
        if save_path is not None:
            fig.savefig(save_path, dpi=100)
        return fig

    # ── VIZ 4: ΔQ Convergence ────────────────────────────────────

    def plot_convergence(self, save_path: Optional[str] = None) -> MplFigure:
        dq       = self.agent.episode_delta_q
        smoothed = pd.Series(dq).rolling(20, min_periods=1).mean().to_numpy()

        fig, ax = plt.subplots(figsize=(12, 5))
        ax.semilogy(dq,       color='lightblue', alpha=0.5, linewidth=1,   label='Raw ΔQ')
        ax.semilogy(smoothed, color='steelblue', linewidth=2.0,             label='Smoothed ΔQ')
        ax.axhline(CONFIG['conv_threshold'], color='red', linestyle='--',
                   label=f'Threshold={CONFIG["conv_threshold"]}')
        ax.set_title(f'{CONFIG["watermark"]} — Q-Table Convergence (ΔQ per Episode)',
                     fontsize=13, fontweight='bold')
        ax.set_xlabel('Episode')
        ax.set_ylabel('Mean |ΔQ| (log scale)')
        ax.legend()
        plt.tight_layout()
        if save_path is not None:
            fig.savefig(save_path, dpi=100)
        return fig

    # ── VIZ 5: Mastery Progression ───────────────────────────────

    def plot_mastery_progression(self,
                                  student_id: Optional[str] = None,
                                  save_path:  Optional[str] = None) -> MplFigure:
        sid   = student_id if student_id is not None else self.students[0].student_id
        sdata = self.df[self.df['student_id'] == sid].sort_values('episode')
        if sdata.empty:
            sdata = self.df.groupby('episode')['new_mastery'].mean().reset_index()
            title_suffix = "(Average)"
        else:
            title_suffix = f"({sid})"

        fig, ax = plt.subplots(figsize=(12, 5))
        if 'concept_id' in sdata.columns:
            for cid in sdata['concept_id'].unique():
                cd = sdata[sdata['concept_id'] == cid].sort_values('episode')
                # ── FIX 9: .to_numpy() avoids pandas ExtensionArray plot error ──
                ax.plot(cd['episode'].to_numpy(), cd['new_mastery'].to_numpy(),
                        label=f"{cid}: {CONCEPTS[cid]['name'][:14]}",
                        alpha=0.8, linewidth=1.5)
        else:
            ax.plot(sdata['episode'].to_numpy(),
                    sdata['new_mastery'].to_numpy(), linewidth=2)
        ax.axhline(CONFIG['mastery_threshold'], linestyle='--', color='green',
                   alpha=0.7, label=f'Threshold ({CONFIG["mastery_threshold"]})')
        ax.set_title(f'{CONFIG["watermark"]} — Mastery Progression {title_suffix}',
                     fontsize=13, fontweight='bold')
        ax.set_xlabel('Episode')
        ax.set_ylabel('Mastery Score')
        ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=7)
        plt.tight_layout()
        if save_path is not None:
            fig.savefig(save_path, dpi=100, bbox_inches='tight')
        return fig

    # ── VIZ 6: Utility Bar Chart ──────────────────────────────────

    def plot_utility_bars(self,
                           student:   StudentProfile,
                           save_path: Optional[str] = None) -> MplFigure:
        valid  = self.kg.get_valid_actions(student.mastery_vector)
        ranked = self.utility.rank_actions(student, valid)

        names  = [CONCEPTS[a]['name'][:18] for a, _ in ranked]
        scores = [float(s) for _, s in ranked]
        colors = ['#4D96FF' if i == 0 else '#A8C8FF' for i in range(len(scores))]

        fig, ax = plt.subplots(figsize=(12, 5))
        ax.barh(names[::-1], scores[::-1], color=colors[::-1])
        ax.set_xlabel('Utility Score')
        ax.set_title(f'{CONFIG["watermark"]} — Utility Scores for {student.student_id}',
                     fontsize=13, fontweight='bold')
        ax.axvline(0.5, linestyle='--', color='gray', alpha=0.5)
        for i, (name, score) in enumerate(zip(names[::-1], scores[::-1])):
            ax.text(score + 0.01, i, f'{score:.3f}', va='center', fontsize=9)
        plt.tight_layout()
        if save_path is not None:
            fig.savefig(save_path, dpi=100)
        return fig

    # ── VIZ 7: ε Decay ───────────────────────────────────────────

    def plot_epsilon_decay(self, save_path: Optional[str] = None) -> MplFigure:
        return plot_exploration(self.tracker, self.agent, save_path)

    # ── VIZ 8: Mastery Rate Metric ───────────────────────────────

    def plot_mastery_rate(self, save_path: Optional[str] = None) -> MplFigure:
        avg_mr: List[float] = []
        thr = CONFIG['mastery_threshold']
        for ep in sorted(self.df['episode'].unique()):
            ep_data = self.df[self.df['episode'] == ep]
            mr = float((ep_data['new_mastery'] >= thr).mean() * 100)
            avg_mr.append(mr)

        fig, ax = plt.subplots(figsize=(12, 5))
        ax.plot(avg_mr, color='mediumseagreen', linewidth=2.5)
        ax.fill_between(range(len(avg_mr)), avg_mr, alpha=0.2, color='mediumseagreen')
        ax.set_title(f'{CONFIG["watermark"]} — Mastery Rate (MR) over Episodes',
                     fontsize=13, fontweight='bold')
        ax.set_xlabel('Episode')
        ax.set_ylabel('Mastery Rate (%)')
        plt.tight_layout()
        if save_path is not None:
            fig.savefig(save_path, dpi=100)
        return fig

    # ── VIZ 9: Satisfaction Trend ─────────────────────────────────

    def plot_satisfaction_trend(self, save_path: Optional[str] = None) -> MplFigure:
        avg_sat = self.df.groupby('episode')['satisfaction'].mean()

        fig, ax = plt.subplots(figsize=(12, 5))
        # ── FIX 9 (continued): .to_numpy() for pandas index/values ──
        ax.plot(avg_sat.index.to_numpy(), avg_sat.to_numpy(),
                color='darkorchid', linewidth=2.5)
        ax.fill_between(avg_sat.index.to_numpy(), avg_sat.to_numpy(),
                        alpha=0.15, color='darkorchid')
        ax.set_title(f'{CONFIG["watermark"]} — Student Satisfaction Score Trend',
                     fontsize=13, fontweight='bold')
        ax.set_xlabel('Episode')
        ax.set_ylabel('Avg Satisfaction Score')
        ax.set_ylim(0, 1.05)
        plt.tight_layout()
        if save_path is not None:
            fig.savefig(save_path, dpi=100)
        return fig

    # ── VIZ 10: Learning Efficiency Comparison ───────────────────

    def plot_learning_efficiency(self,
                                  baseline_le: float,
                                  pathwise_le: float,
                                  save_path:   Optional[str] = None) -> MplFigure:
        fig, ax = plt.subplots(figsize=(8, 5))
        bars = ax.bar(['Random Baseline', 'PathWise AI'],
                      [baseline_le, pathwise_le],
                      color=['#FF6B6B', '#6BCB77'],
                      edgecolor='black', width=0.5)
        for bar, val in zip(bars, [baseline_le, pathwise_le]):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.3,
                    f'{val:.2f}%', ha='center', va='bottom',
                    fontsize=12, fontweight='bold')
        improvement = (pathwise_le - baseline_le) / baseline_le * 100
        ax.set_title(f'{CONFIG["watermark"]} — Learning Efficiency\n'
                     f'Improvement: +{improvement:.1f}% over baseline',
                     fontsize=13, fontweight='bold')
        ax.set_ylabel('Learning Efficiency (concepts/hour × 100)')
        ax.set_ylim(0, max(baseline_le, pathwise_le) * 1.3)
        plt.tight_layout()
        if save_path is not None:
            fig.savefig(save_path, dpi=100)
        return fig

    def render_all(self,
                    student:    Optional[StudentProfile] = None,
                    output_dir: str = '.') -> None:
        """Render and save all 10 visualisations."""
        os.makedirs(output_dir, exist_ok=True)
        s = student if student is not None else self.students[0]
        self.plot_knowledge_graph(s,             save_path=f'{output_dir}/01_knowledge_graph.png')
        self.plot_q_heatmap(                      save_path=f'{output_dir}/02_q_heatmap.png')
        self.plot_reward_curve(                   save_path=f'{output_dir}/03_reward_curve.png')
        self.plot_convergence(                    save_path=f'{output_dir}/04_convergence.png')
        self.plot_mastery_progression(s.student_id, save_path=f'{output_dir}/05_mastery_prog.png')
        self.plot_utility_bars(s,                 save_path=f'{output_dir}/06_utility_bars.png')
        self.plot_epsilon_decay(                  save_path=f'{output_dir}/07_epsilon_decay.png')
        self.plot_mastery_rate(                   save_path=f'{output_dir}/08_mastery_rate.png')
        self.plot_satisfaction_trend(             save_path=f'{output_dir}/09_satisfaction.png')

        _ev      = EvaluationEngine(self.students, self.df, self.utility, self.kg)
        _metrics = _ev.full_report()
        self.plot_learning_efficiency(
            _metrics['baseline_le'], _metrics['pathwise_le'],
            save_path=f'{output_dir}/10_learning_efficiency.png',
        )
        print(f"✅ All 10 visualisations saved to {output_dir}/")


print("✅ AnalyticsEngine defined (10 visualisations)")

✅ AnalyticsEngine defined (10 visualisations)


In [28]:
class EvaluationEngine:
    """
    Computes the three core PathWise AI metrics:
        LE  = (concepts_mastered / total_study_hours) × 100
        MR  = (concepts with mastery ≥ 0.7 / 15)     × 100
        SS  = Σ(utility_alignment_score) / n_recs
    """

    def __init__(self,
                 students: List[StudentProfile],
                 dataset:  pd.DataFrame,
                 utility:  UtilityCalculator,
                 kg:       KnowledgeGraph) -> None:
        self.students = students
        self.df       = dataset
        self.utility  = utility
        self.kg       = kg

    def learning_efficiency(self,
                             recs:        List[Dict[str, Any]],
                             total_hours: float) -> float:
        mastered = sum(1 for r in recs
                       if float(r.get('new_mastery', 0)) >= CONFIG['mastery_threshold'])
        if total_hours < 1e-6:
            return 0.0
        return (mastered / total_hours) * 100

    def mastery_rate(self, student: StudentProfile) -> float:
        mastered = int(np.sum(student.mastery_vector >= CONFIG['mastery_threshold']))
        return (mastered / N_CONCEPTS) * 100

    def satisfaction_score(self,
                            recommendations: List[Dict[str, Any]],
                            student: StudentProfile) -> float:
        if not recommendations:
            return 0.0
        scores = [float(r.get('utility_score', 0.0)) for r in recommendations]
        return float(np.mean(scores))

    def baseline_le(self,
                     n_episodes: int = CONFIG['n_learning_episodes']) -> float:
        """Random policy LE."""
        rng_b          = np.random.default_rng(CONFIG['random_seed'] + 99)
        total_mastered = 0
        total_hours    = 0.0
        for s in self.students[:50]:
            m = s.prior_knowledge_vector.copy()
            for _ in range(n_episodes):
                cid        = str(rng_b.choice(CONCEPT_IDS))
                idx        = CONCEPT_INDEX[cid]
                m[idx]     = min(1.0, m[idx] + 0.05)
                total_hours += float(CONCEPTS[cid]['est_hours'])
            total_mastered += int(np.sum(m >= CONFIG['mastery_threshold']))
        return (total_mastered / total_hours) * 100 if total_hours > 0 else 0.0

    def pathwise_le(self) -> float:
        total_m = 0
        total_h = 0.0
        for s in self.students[:50]:
            total_m += int(np.sum(s.mastery_vector >= CONFIG['mastery_threshold']))
            total_h += s.available_study_hours * CONFIG['n_learning_episodes']
        return (total_m / total_h) * 100 if total_h > 0 else 0.0

    def full_report(self) -> Dict[str, Any]:
        b_le       = self.baseline_le()
        p_le       = self.pathwise_le()
        avg_mr     = float(np.mean([self.mastery_rate(s) for s in self.students]))
        avg_sat    = float(self.df['satisfaction'].mean())
        improvement = (p_le - b_le) / b_le * 100 if b_le > 0 else 0.0
        return {
            'baseline_le':          round(b_le, 3),
            'pathwise_le':          round(p_le, 3),
            'le_improvement_pct':   round(improvement, 1),
            'avg_mastery_rate':     round(avg_mr, 2),
            'avg_satisfaction':     round(avg_sat, 4),
        }


print("✅ EvaluationEngine defined")

✅ EvaluationEngine defined


In [29]:
def run_unit_tests(kg:       KnowledgeGraph,
                   bn:       BayesianEngine,
                   agent:    QLearningAgent,
                   utility:  UtilityCalculator,
                   students: List[StudentProfile]) -> bool:
    """Run all unit / functional / edge-case tests."""
    passed = 0
    failed = 0

    def check(name: str, condition: bool, msg: str = "") -> None:
        nonlocal passed, failed
        if not condition:
            failed += 1
            print(f"  ❌ [FAIL] {name}: {msg}")
        else:
            passed += 1
            print(f"  ✅ [PASS] {name}")

    print("\n── KnowledgeGraph Tests ──")
    check("KG has 15 nodes",       len(kg.graph.nodes()) == 15)
    check("KG has correct edges",  len(kg.graph.edges()) == len(PREREQUISITES))
    check("C15 has no successors", len(list(kg.graph.successors('C15'))) == 0)
    check("C01 satisfiable zero mastery",
          kg.prerequisites_satisfied('C01', np.zeros(N_CONCEPTS)))
    check("C15 blocked zero mastery",
          not kg.prerequisites_satisfied('C15', np.zeros(N_CONCEPTS)))

    print("\n── BayesianNetwork Tests ──")
    s0   = students[0]
    prob = bn.p_mastery_high(s0, 'C01')
    check("BN mastery prob ∈ [0,1]", 0.0 <= prob <= 1.0)
    sp   = bn.p_success_pass(s0, 'C07')
    check("BN success prob ∈ [0,1]", 0.0 <= sp <= 1.0)
    result = bn.infer_mastery(s0, 'C01')
    mp_sum = float(np.sum(result['MasteryProbability']))
    check("BN MasteryProbability sums to 1",  abs(mp_sum - 1.0) < 0.01, f"got {mp_sum}")
    sp_sum = float(np.sum(result['SuccessProbability']))
    check("BN SuccessProbability sums to 1",  abs(sp_sum - 1.0) < 0.01, f"got {sp_sum}")

    print("\n── Utility Tests ──")
    u_val = utility.compute(s0, 'C01')
    check("Utility ∈ [0,1]", 0.0 <= u_val <= 1.0)
    w_sum = (CONFIG['wu_mastery'] + CONFIG['wu_time'] +
             CONFIG['wu_preference'] + CONFIG['wu_goal'])
    check("Utility weights sum to 1.0", abs(w_sum - 1.0) < 1e-9)
    ranked = utility.rank_actions(s0, ['C01', 'C02', 'C04'])
    check("Utility ranking non-empty", len(ranked) > 0)

    print("\n── Q-Learning Tests ──")
    rng_t = np.random.default_rng(0)
    st    = kg.get_valid_actions(s0.mastery_vector)
    _, flag_explore  = agent.select_action(0, st, epsilon=1.0, rng=rng_t)
    check("Epsilon=1 always explores",  flag_explore)
    _, flag_exploit = agent.select_action(0, st, epsilon=0.0, rng=rng_t)
    check("Epsilon=0 always exploits",  not flag_exploit)
    dq = agent.update(0, 'C01', 0.5, 1, ['C01'], t=1)
    check("Q-update returns non-negative |ΔQ|", dq >= 0)

    print("\n── Edge Cases ──")
    full_m    = np.ones(N_CONCEPTS)
    full_m[0] = 0.0
    valid_all_mastered = kg.get_valid_actions(full_m)
    check("C15 not in valid when prereqs unmet", 'C15' not in valid_all_mastered)
    check("Fallback returns at least one action", len(kg.get_valid_actions(np.ones(N_CONCEPTS))) >= 1)

    print(f"\n── Results: {passed} passed / {failed} failed ──")
    return failed == 0


print("✅ Unit tests defined")

✅ Unit tests defined


In [30]:
def save_artefacts(agent:      QLearningAgent,
                   tracker:    ExplorationTracker,
                   students:   List[StudentProfile],
                   metrics:    Dict[str, Any],
                   output_dir: str = '.') -> None:
    """Pickle Q-table and save JSON artefacts for Streamlit app."""
    os.makedirs(output_dir, exist_ok=True)

    with open(f'{output_dir}/q_table.pkl', 'wb') as f:
        pickle.dump(dict(agent.Q), f)

    student_data = [
        {
            'student_id':          s.student_id,
            'mastery_vector':      s.mastery_vector.tolist(),
            'prior_knowledge':     s.prior_knowledge_vector.tolist(),
            'learning_preference': s.learning_preference,
            'learning_goal':       s.learning_goal,
            'available_hours':     s.available_study_hours,
            'satisfaction':        s.satisfaction_score,
            'completion_rate':     s.completion_rate,
        }
        for s in students
    ]
    with open(f'{output_dir}/students.json', 'w') as f:
        json.dump(student_data, f, indent=2)

    with open(f'{output_dir}/metrics.json', 'w') as f:
        json.dump(metrics, f, indent=2)

    training_history: Dict[str, Any] = {
        'episode_rewards':   agent.episode_rewards,
        'episode_delta_q':   agent.episode_delta_q,
        'converged_at':      agent.converged_at,
        'epsilon_history':   tracker.epsilon_history[:1000],
        'stagnation_events': tracker.stagnation_events,
    }
    with open(f'{output_dir}/training_history.json', 'w') as f:
        json.dump(training_history, f, indent=2)

    print(f"✅ Artefacts saved to {output_dir}/")


# ═══════════════════════════════════════════════════════════════════
# MAIN — Orchestration (Run All in Colab)
# ═══════════════════════════════════════════════════════════════════

if __name__ == '__main__' or True:
    print("\n" + "═"*65)
    print("  PathWise AI — Training Pipeline Starting")
    print("═"*65)

    kg       = KnowledgeGraph()
    print(f"Step 1 ✅ KG: {len(kg.graph.nodes())} nodes, {len(kg.graph.edges())} edges")

    bn       = BayesianEngine()
    print(f"Step 2 ✅ BN: pgmpy={'yes' if bn.model else 'fallback'}")

    rng      = np.random.default_rng(CONFIG['random_seed'])
    data_gen = DataGenerator(rng)
    students = data_gen.generate_students(CONFIG['n_students'])
    dataset  = data_gen.build_dataset(students, kg)
    print(f"Step 3 ✅ Data: {len(students)} students × "
          f"{CONFIG['n_learning_episodes']} episodes = {len(dataset)} rows")

    mdp     = MDPEnvironment(kg, bn)
    utility = UtilityCalculator(bn)
    print("Step 4 ✅ MDP + Utility ready")

    print(f"\nStep 5 — Q-Learning: {CONFIG['n_train_episodes']} episodes...")
    agent, tracker = run_training(students, kg, bn, mdp, data_gen)
    print(f"Step 5 ✅ Training complete | Q-table entries: {len(agent.Q)}")

    rec_eng  = RecommendationEngine(agent, kg, bn, utility, mdp)
    print("Step 6 ✅ RecommendationEngine ready")

    analytics = AnalyticsEngine(agent, tracker, students, kg, bn, utility, rec_eng, dataset)
    evaluator = EvaluationEngine(students, dataset, utility, kg)
    metrics   = evaluator.full_report()
    print(f"Step 7 ✅ Metrics: {metrics}")

    print("\nStep 8 — Unit Tests:")
    run_unit_tests(kg, bn, agent, utility, students)

    analytics.render_all(students[0], output_dir='./pathwise_output')
    save_artefacts(agent, tracker, students, metrics, output_dir='./pathwise_output')

    s_demo = students[0]
    recs   = rec_eng.recommend(s_demo, top_k=3)
    print(f"\n📌 Demo recommendation for {s_demo.student_id}:")
    for i, r in enumerate(recs):
        print(f"   {i+1}. {r['concept_id']} — {r['concept_name']}"
              f"  Q={r['q_value']:.4f}  U={r['utility_score']:.4f}"
              f"  BN_MP={r['bn_mastery_p']:.4f}")

        print("\n" + "═"*65)
    print("  PathWise AI — Training Pipeline Starting")
    print("═"*65)

    kg       = KnowledgeGraph()
    print(f"Step 1 ✅ KG: {len(kg.graph.nodes())} nodes, {len(kg.graph.edges())} edges")

    bn       = BayesianEngine()
    print(f"Step 2 ✅ BN: pgmpy={'yes' if bn.model else 'fallback'}")

    rng      = np.random.default_rng(CONFIG['random_seed'])
    data_gen = DataGenerator(rng)
    students = data_gen.generate_students(CONFIG['n_students'])
    dataset  = data_gen.build_dataset(students, kg)
    print(f"Step 3 ✅ Data: {len(students)} students × "
          f"{CONFIG['n_learning_episodes']} episodes = {len(dataset)} rows")

    mdp     = MDPEnvironment(kg, bn)
    utility = UtilityCalculator(bn)
    print("Step 4 ✅ MDP + Utility ready")

    print(f"\nStep 5 — Q-Learning: {CONFIG['n_train_episodes']} episodes...")
    agent, tracker = run_training(students, kg, bn, mdp, data_gen)
    print(f"Step 5 ✅ Training complete | Q-table entries: {len(agent.Q)}")

    rec_eng  = RecommendationEngine(agent, kg, bn, utility, mdp)
    print("Step 6 ✅ RecommendationEngine ready")

    analytics = AnalyticsEngine(agent, tracker, students, kg, bn, utility, rec_eng, dataset)
    evaluator = EvaluationEngine(students, dataset, utility, kg)
    metrics   = evaluator.full_report()
    print(f"Step 7 ✅ Metrics: {metrics}")

    print("\nStep 8 — Unit Tests:")
    run_unit_tests(kg, bn, agent, utility, students)

    analytics.render_all(students[0], output_dir='./pathwise_output')
    save_artefacts(agent, tracker, students, metrics, output_dir='./pathwise_output')

    s_demo = students[0]
    recs   = rec_eng.recommend(s_demo, top_k=3)
    print(f"\n📌 Demo recommendation for {s_demo.student_id}:")
    for i, r in enumerate(recs):
        print(f"   {i+1}. {r['concept_id']} — {r['concept_name']}"
              f"  Q={r['q_value']:.4f}  U={r['utility_score']:.4f}"
              f"  BN_MP={r['bn_mastery_p']:.4f}")

    if COLAB_MODE:
        student_ids = [s.student_id for s in students[:20]]
        dd_student  = widgets.Dropdown(            # type: ignore[union-attr]
            options=student_ids,
            description='Student:',
            value=student_ids[0],
        )
        out_widget = widgets.Output()              # type: ignore[union-attr]

        def on_change(change: Dict[str, Any]) -> None:
            if change['type'] == 'change' and change['name'] == 'value':
                sid  = str(change['new'])
                s    = next(x for x in students if x.student_id == sid)
                recs_w = rec_eng.recommend(s, top_k=3)
                with out_widget:
                    clear_output(wait=True)        # type: ignore[union-attr]
                    print(f"\n📌 Recommendations for {sid} "
                          f"(goal={s.learning_goal}, pref={s.learning_preference}):")
                    for i, r in enumerate(recs_w):
                        print(f"  {i+1}. {r['concept_id']} — {r['concept_name']}")
                        print(f"      Q={r['q_value']:.4f} | "
                              f"Utility={r['utility_score']:.4f} | "
                              f"BN_mastery={r['bn_mastery_p']:.4f} | "
                              f"BN_success={r['bn_success_p']:.4f}")
                    fig = analytics.plot_knowledge_graph(s)
                    plt.show()
                    plt.close()

        dd_student.observe(on_change)
        on_change({'type': 'change', 'name': 'value', 'new': student_ids[0]})
        display(                                   # type: ignore[union-attr]
            widgets.VBox([dd_student, out_widget]) # type: ignore[union-attr]
        )

    print("\n" + "═"*65)
    print("  PathWise AI — Pipeline Complete ✅")
    print("═"*65)

2026-06-10 16:02:29,997 [PathWiseAI] INFO: Bayesian Network built and validated with pgmpy



═════════════════════════════════════════════════════════════════
  PathWise AI — Training Pipeline Starting
═════════════════════════════════════════════════════════════════
Step 1 ✅ KG: 15 nodes, 27 edges
Step 2 ✅ BN: pgmpy=yes


2026-06-10 16:02:30,549 [PathWiseAI] INFO: Episode    0 | ε=1.000 | reward=8.089 | ΔQ=0.1785


Step 3 ✅ Data: 200 students × 20 episodes = 4000 rows
Step 4 ✅ MDP + Utility ready

Step 5 — Q-Learning: 500 episodes...


2026-06-10 16:02:32,630 [PathWiseAI] INFO: Episode  100 | ε=0.607 | reward=8.240 | ΔQ=0.0775
2026-06-10 16:02:34,460 [PathWiseAI] INFO: Episode  200 | ε=0.368 | reward=7.560 | ΔQ=0.0403
2026-06-10 16:02:36,176 [PathWiseAI] INFO: Episode  300 | ε=0.223 | reward=7.035 | ΔQ=0.0247
2026-06-10 16:02:37,703 [PathWiseAI] INFO: Episode  400 | ε=0.135 | reward=7.365 | ΔQ=0.0227


...

2026-06-10 16:02:42,670 [PathWiseAI] INFO: Episode    0 | ε=1.000 | reward=8.089 | ΔQ=0.1785


Step 3 ✅ Data: 200 students × 20 episodes = 4000 rows
Step 4 ✅ MDP + Utility ready

Step 5 — Q-Learning: 500 episodes...


2026-06-10 16:02:44,346 [PathWiseAI] INFO: Episode  100 | ε=0.607 | reward=8.240 | ΔQ=0.0775
2026-06-10 16:02:46,145 [PathWiseAI] INFO: Episode  200 | ε=0.368 | reward=7.560 | ΔQ=0.0403
2026-06-10 16:02:47,791 [PathWiseAI] INFO: Episode  300 | ε=0.223 | reward=7.035 | ΔQ=0.0247
2026-06-10 16:02:49,356 [PathWiseAI] INFO: Episode  400 | ε=0.135 | reward=7.365 | ΔQ=0.0227


Step 5 ✅ Training complete | Q-table entries: 174
Step 6 ✅ RecommendationEngine ready
Step 7 ✅ Metrics: {'baseline_le': 0.072, 'pathwise_le': 1.369, 'le_improvement_pct': 1799.8, 'avg_mastery_rate': 9.63, 'avg_satisfaction': 0.7136}

Step 8 — Unit Tests:

── KnowledgeGraph Tests ──
  ✅ [PASS] KG has 15 nodes
  ✅ [PASS] KG has correct edges
  ✅ [PASS] C15 has no successors
  ✅ [PASS] C01 satisfiable zero mastery
  ✅ [PASS] C15 blocked zero mastery

── BayesianNetwork Tests ──
  ✅ [PASS] BN mastery prob ∈ [0,1]
  ✅ [PASS] BN success prob ∈ [0,1]
  ✅ [PASS] BN MasteryProbability sums to 1
  ✅ [PASS] BN SuccessProbability sums to 1

── Utility Tests ──
  ✅ [PASS] Utility ∈ [0,1]
  ✅ [PASS] Utility weights sum to 1.0
  ✅ [PASS] Utility ranking non-empty

── Q-Learning Tests ──
  ✅ [PASS] Epsilon=1 always explores
  ✅ [PASS] Epsilon=0 always exploits
  ✅ [PASS] Q-update returns non-negative |ΔQ|

── Edge Cases ──
  ✅ [PASS] C15 not in valid when prereqs unmet
  ✅ [PASS] Fallback returns at l


═════════════════════════════════════════════════════════════════
  PathWise AI — Pipeline Complete ✅
═════════════════════════════════════════════════════════════════


In [31]:
# ═══════════════════════════════════════════════════════════════════
# INTERACTIVE DASHBOARD  (ipywidgets multi-tab UI)
# ═══════════════════════════════════════════════════════════════════

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# ── CSS ─────────────────────────────────────────────────────────────
display(HTML("""
<style>
.pw-card{background:#f0f4f8;border-radius:8px;padding:12px 20px;
         margin:5px;display:inline-block;min-width:120px;text-align:center;}
.pw-val{font-size:26px;font-weight:bold;color:#2196F3;}
.pw-lbl{font-size:11px;color:#666;margin-top:4px;}
</style>
"""))

# ── Header ───────────────────────────────────────────────────────────
display(HTML(f"""
<div style="background:linear-gradient(135deg,#667eea,#764ba2);
            padding:22px 28px;border-radius:12px;color:white;margin-bottom:16px;">
  <h1 style="margin:0;font-size:26px;">PathWise AI  — Interactive Dashboard</h1>
  <p  style="margin:6px 0 0 0;opacity:0.9;font-size:14px;">
      Personalized Learning Path Recommender &nbsp;|&nbsp;
      RL · Bayesian · Utility Theory · Knowledge Graph</p>
</div>
<div style="margin-bottom:18px;">
  <div class="pw-card">
    <div class="pw-val" style="color:#2e7d32;">{metrics['avg_mastery_rate']:.1f}%</div>
    <div class="pw-lbl">Avg Mastery Rate</div>
  </div>
  <div class="pw-card">
    <div class="pw-val" style="color:#1565c0;">{metrics['avg_satisfaction']:.3f}</div>
    <div class="pw-lbl">Avg Satisfaction</div>
  </div>
  <div class="pw-card">
    <div class="pw-val" style="color:#e65100;">+{metrics['le_improvement_pct']:.1f}%</div>
    <div class="pw-lbl">LE Gain vs Baseline</div>
  </div>
  <div class="pw-card">
    <div class="pw-val" style="color:#6a1b9a;">{agent.converged_at or 'N/A'}</div>
    <div class="pw-lbl">Convergence Episode</div>
  </div>
  <div class="pw-card">
    <div class="pw-val" style="color:#00695c;">{len(agent.Q)}</div>
    <div class="pw-lbl">Q-Table States</div>
  </div>
  <div class="pw-card">
    <div class="pw-val" style="color:#b71c1c;">{CONFIG['n_students']}</div>
    <div class="pw-lbl">Students</div>
  </div>
</div>
"""))

student_ids_all = [s.student_id for s in students]

# ════════════════════════════════════════════════════════════════════
# TAB 1 — Student Explorer
# ════════════════════════════════════════════════════════════════════
out_s = widgets.Output()
dd_sid  = widgets.Dropdown(options=student_ids_all, description='Student:',
                            style={'description_width':'80px'}, layout=widgets.Layout(width='160px'))
dd_goal = widgets.Dropdown(options=['(keep)','speed','depth','certification'],
                            description='Goal Override:',
                            style={'description_width':'110px'}, layout=widgets.Layout(width='200px'))
dd_pref = widgets.Dropdown(options=['(keep)','visual','reading','practice'],
                            description='Pref Override:',
                            style={'description_width':'110px'}, layout=widgets.Layout(width='200px'))
btn_s   = widgets.Button(description='Recommend', button_style='primary',
                          layout=widgets.Layout(width='120px'))
ctrl_s  = widgets.HBox([dd_sid, dd_goal, dd_pref, btn_s])

def _show_student(_=None):
    sid   = dd_sid.value
    s_raw = next(x for x in students if x.student_id == sid)
    s     = copy.deepcopy(s_raw)
    if dd_goal.value != '(keep)': s.learning_goal       = dd_goal.value
    if dd_pref.value != '(keep)': s.learning_preference = dd_pref.value
    recs_w = rec_eng.recommend(s, top_k=5)
    n_mastered = int(np.sum(s.mastery_vector >= CONFIG['mastery_threshold']))
    with out_s:
        clear_output(wait=True)
        display(HTML(f"""
        <div style="background:#e8f4fd;border-radius:8px;padding:10px 14px;margin:8px 0;">
          <b>{sid}</b> &nbsp;|&nbsp; Goal: <b>{s.learning_goal}</b> &nbsp;|&nbsp;
          Pref: <b>{s.learning_preference}</b> &nbsp;|&nbsp;
          Hours/day: <b>{s.available_study_hours:.1f}</b> &nbsp;|&nbsp;
          Mastered: <b>{n_mastered}/15</b> &nbsp;|&nbsp;
          Satisfaction: <b>{s.satisfaction_score:.3f}</b>
        </div>"""))
        # Recommendations table
        rows = ''.join(f"""
        <tr style="background:{'#f0f8ff' if i==0 else 'white' if i%2==0 else '#f9f9f9'};">
          <td style="padding:7px;font-weight:bold;">#{i+1}</td>
          <td style="padding:7px;"><b>{r['concept_id']}</b><br>
              <small style="color:#555;">{r['concept_name']}</small></td>
          <td style="padding:7px;text-align:center;">{r['q_value']:.4f}</td>
          <td style="padding:7px;text-align:center;">{r['utility_score']:.4f}</td>
          <td style="padding:7px;text-align:center;">{r['bn_mastery_p']:.3f}</td>
          <td style="padding:7px;text-align:center;">{r['bn_success_p']:.3f}</td>
          <td style="padding:7px;text-align:center;">{r['est_hours']}h</td>
          <td style="padding:7px;text-align:center;">{'✅' if r['prereq_ok'] else '⚠️'}</td>
        </tr>""" for i, r in enumerate(recs_w))
        display(HTML(f"""
        <table style="width:100%;border-collapse:collapse;margin:8px 0;font-size:13px;">
          <tr style="background:#667eea;color:white;">
            <th style="padding:8px;">Rank</th><th style="padding:8px;">Concept</th>
            <th style="padding:8px;">Q-Value</th><th style="padding:8px;">Utility</th>
            <th style="padding:8px;">BN Mastery P</th><th style="padding:8px;">BN Success P</th>
            <th style="padding:8px;">Est.Hours</th><th style="padding:8px;">Prereqs</th>
          </tr>{rows}</table>"""))
        # Mastery bar chart
        fig, ax = plt.subplots(figsize=(13, 4))
        names  = [CONCEPTS[c]['name'][:15] for c in CONCEPT_IDS]
        mvals  = s.mastery_vector
        colors = [CONFIG['clr_mastered'] if m>=CONFIG['mastery_threshold']
                  else CONFIG['clr_in_progress'] if m>0.1 else CONFIG['clr_not_started']
                  for m in mvals]
        ax.barh(names[::-1], mvals[::-1], color=colors[::-1], edgecolor='#aaa', linewidth=0.5)
        ax.axvline(CONFIG['mastery_threshold'], linestyle='--', color='navy', alpha=0.7,
                   label=f'Mastery threshold ({CONFIG["mastery_threshold"]})')
        for i, v in enumerate(mvals[::-1]):
            ax.text(v+0.01, i, f'{v:.2f}', va='center', fontsize=8)
        ax.set_xlim(0, 1.15); ax.set_xlabel('Mastery Level')
        ax.set_title(f'Mastery Profile — {sid}  ({n_mastered}/15 mastered)', fontweight='bold')
        ax.legend(fontsize=9); plt.tight_layout(); plt.show(); plt.close()

btn_s.on_click(_show_student)
dd_sid.observe(lambda c: _show_student() if c['name']=='value' else None, names='value')
_show_student()
tab1 = widgets.VBox([ctrl_s, out_s])

# ════════════════════════════════════════════════════════════════════
# TAB 2 — Knowledge Graph
# ════════════════════════════════════════════════════════════════════
out_kg  = widgets.Output()
dd_kg   = widgets.Dropdown(options=student_ids_all, description='Student:',
                             style={'description_width':'80px'}, layout=widgets.Layout(width='160px'))
btn_kg  = widgets.Button(description='Show Graph', button_style='info',
                          layout=widgets.Layout(width='120px'))

def _show_kg(_=None):
    sid  = dd_kg.value
    s    = next(x for x in students if x.student_id == sid)
    recs_k = rec_eng.recommend(s, top_k=1)
    rec_id = recs_k[0]['concept_id'] if recs_k else None
    with out_kg:
        clear_output(wait=True)
        fig = analytics.plot_knowledge_graph(s, recommended=rec_id)
        plt.show(); plt.close()
        valid    = kg.get_valid_actions(s.mastery_vector)
        n_m      = int(np.sum(s.mastery_vector >= CONFIG['mastery_threshold']))
        # Suggested path: topological order of un-mastered nodes
        topo  = kg.topological_order()
        path  = [c for c in topo if s.mastery_vector[CONCEPT_INDEX[c]] < CONFIG['mastery_threshold']][:5]
        path_str = ' → '.join(f"{c}({CONCEPTS[c]['name'][:10]})" for c in path)
        display(HTML(f"""
        <div style="background:#e8f5e9;border-radius:8px;padding:10px 14px;margin:6px 0;">
          <b>Progress:</b> {n_m}/15 mastered ({n_m/N_CONCEPTS*100:.0f}%) &nbsp;|&nbsp;
          <b>Recommended next:</b> {rec_id} — {CONCEPTS[rec_id]['name'] if rec_id else 'N/A'}<br>
          <b>Suggested learning path:</b> {path_str}
        </div>"""))

btn_kg.on_click(_show_kg)
dd_kg.observe(lambda c: _show_kg() if c['name']=='value' else None, names='value')
_show_kg()
tab2 = widgets.VBox([widgets.HBox([dd_kg, btn_kg]), out_kg])

# ════════════════════════════════════════════════════════════════════
# TAB 3 — RL Policy
# ════════════════════════════════════════════════════════════════════
out_rl = widgets.Output()
with out_rl:
    for plot_fn, title in [
        (analytics.plot_reward_curve,  'Reward Curve'),
        (analytics.plot_convergence,   'Convergence'),
        (analytics.plot_q_heatmap,     'Q-Value Heatmap'),
        (analytics.plot_epsilon_decay, 'Exploration Policy'),
    ]:
        fig = plot_fn()
        plt.show(); plt.close()
    display(HTML(f"""
    <div style="background:#fff3e0;border-radius:8px;padding:14px;margin:10px 0;">
      <h4 style="margin:0 0 8px 0;">RL Hyper-parameter Summary</h4>
      <table style="border-collapse:collapse;font-size:13px;">
        <tr><td style="padding:4px 12px;"><b>Episodes</b></td><td>{CONFIG['n_train_episodes']}</td>
            <td style="padding:4px 24px;"><b>Converged</b></td><td>{agent.converged_at or 'ongoing'}</td></tr>
        <tr><td style="padding:4px 12px;"><b>Q-States</b></td><td>{len(agent.Q)}</td>
            <td style="padding:4px 24px;"><b>γ (discount)</b></td><td>{CONFIG['gamma']}</td></tr>
        <tr><td style="padding:4px 12px;"><b>α₀</b></td><td>{CONFIG['alpha_0']}</td>
            <td style="padding:4px 24px;"><b>ε range</b></td><td>{CONFIG['eps_min']}–{CONFIG['eps_max']}</td></tr>
        <tr><td style="padding:4px 12px;"><b>Stagnations</b></td><td>{len(tracker.stagnation_events)}</td>
            <td style="padding:4px 24px;"><b>ε boost</b></td><td>+{CONFIG['eps_boost']}</td></tr>
      </table>
    </div>"""))
tab3 = widgets.VBox([out_rl])

# ════════════════════════════════════════════════════════════════════
# TAB 4 — Learning Analytics
# ════════════════════════════════════════════════════════════════════
out_an = widgets.Output()
with out_an:
    for fn in [analytics.plot_mastery_rate, analytics.plot_satisfaction_trend]:
        fig = fn(); plt.show(); plt.close()
    _ev2     = EvaluationEngine(students, dataset, utility, kg)
    _m2      = _ev2.full_report()
    fig = analytics.plot_learning_efficiency(_m2['baseline_le'], _m2['pathwise_le'])
    plt.show(); plt.close()
    display(HTML(f"""
    <div style="margin:10px 0;">
      <div class="pw-card" style="background:#e8f5e9;">
        <div class="pw-val" style="color:#2e7d32;">{_m2['avg_mastery_rate']:.1f}%</div>
        <div class="pw-lbl">Avg Mastery Rate</div>
      </div>
      <div class="pw-card" style="background:#e3f2fd;">
        <div class="pw-val" style="color:#1565c0;">{_m2['avg_satisfaction']:.3f}</div>
        <div class="pw-lbl">Avg Satisfaction</div>
      </div>
      <div class="pw-card" style="background:#fce4ec;">
        <div class="pw-val" style="color:#880e4f;">{_m2['baseline_le']:.2f}</div>
        <div class="pw-lbl">Baseline LE</div>
      </div>
      <div class="pw-card" style="background:#e8f5e9;">
        <div class="pw-val" style="color:#2e7d32;">{_m2['pathwise_le']:.2f}</div>
        <div class="pw-lbl">PathWise LE</div>
      </div>
      <div class="pw-card" style="background:#fff3e0;">
        <div class="pw-val" style="color:#e65100;">+{_m2['le_improvement_pct']:.1f}%</div>
        <div class="pw-lbl">LE Improvement</div>
      </div>
    </div>
    <p style="font-size:12px;color:#555;">
      LE = (concepts mastered / total study hours) × 100 &nbsp;|&nbsp;
      MR = (concepts ≥ 0.70 mastery / 15) × 100 &nbsp;|&nbsp;
      SS = mean utility alignment score across recommendations
    </p>"""))
tab4 = widgets.VBox([out_an])

# ════════════════════════════════════════════════════════════════════
# TAB 5 — Mastery Progression Explorer
# ════════════════════════════════════════════════════════════════════
out_mp  = widgets.Output()
dd_mp   = widgets.Dropdown(options=student_ids_all, description='Student:',
                             style={'description_width':'80px'}, layout=widgets.Layout(width='160px'))

def _show_mp(_=None):
    sid = dd_mp.value
    with out_mp:
        clear_output(wait=True)
        fig = analytics.plot_mastery_progression(sid)
        plt.show(); plt.close()
        fig = analytics.plot_utility_bars(next(x for x in students if x.student_id==sid))
        plt.show(); plt.close()

dd_mp.observe(lambda c: _show_mp() if c['name']=='value' else None, names='value')
_show_mp()
tab5 = widgets.VBox([dd_mp, out_mp])

# ════════════════════════════════════════════════════════════════════
# Assemble tab panel
# ════════════════════════════════════════════════════════════════════
tabs = widgets.Tab(children=[tab1, tab2, tab3, tab4, tab5])
for i, name in enumerate([
    'Student Explorer', 'Knowledge Graph',
    'RL Policy', 'Analytics', 'Mastery Progression'
]):
    tabs.set_title(i, name)

display(tabs)

In [32]:
# ═══════════════════════════════════════════════════════════════════
# MODULE 1 — Realistic Learner Population + Curriculum Simulation
# ═══════════════════════════════════════════════════════════════════
# The cohort is built from FOUR distinct, research-inspired learner
# archetypes so that every downstream visual shows clear, separable
# behaviour instead of uniform noise.
# -------------------------------------------------------------------
#   Fast Learner          high ability, high availability  → fast mastery
#   Steady Learner        average ability, average hours    → the bulk
#   Working Professional  capable but time-poor             → slow but sure
#   Struggler             low ability, weak foundation      → needs support
# -------------------------------------------------------------------

LEARNER_ARCHETYPES: Dict[str, Dict[str, Any]] = {
    'Fast Learner': {
        'share': 0.15, 'lr': (0.28, 0.035), 'hours': (5.0, 8.0),
        'prior_n': (3, 5), 'prior_lvl': (0.45, 0.75),
        'goals': ['depth', 'certification', 'depth'],
        'age': (20, 28), 'edu': ['Postgraduate', 'Undergraduate'],
        'days': (5, 7),
    },
    'Steady Learner': {
        'share': 0.37, 'lr': (0.18, 0.030), 'hours': (3.5, 6.0),
        'prior_n': (2, 3), 'prior_lvl': (0.25, 0.55),
        'goals': ['certification', 'depth', 'speed'],
        'age': (21, 30), 'edu': ['Undergraduate', 'Postgraduate'],
        'days': (4, 6),
    },
    'Working Professional': {
        'share': 0.27, 'lr': (0.21, 0.030), 'hours': (1.5, 3.5),
        'prior_n': (2, 4), 'prior_lvl': (0.30, 0.60),
        'goals': ['certification', 'speed', 'certification'],
        'age': (27, 42), 'edu': ['Postgraduate', 'Undergraduate'],
        'days': (3, 5),
    },
    'Struggler': {
        'share': 0.21, 'lr': (0.10, 0.020), 'hours': (2.0, 4.5),
        'prior_n': (1, 2), 'prior_lvl': (0.10, 0.35),
        'goals': ['speed', 'certification', 'speed'],
        'age': (19, 36), 'edu': ['Undergraduate', 'High School'],
        'days': (2, 4),
    },
}

# Consistent colour per archetype — reused by every chart in the dashboard.
ARCHETYPE_COLORS: Dict[str, str] = {
    'Fast Learner':         '#2e7d32',   # green
    'Steady Learner':       '#1565c0',   # blue
    'Working Professional': '#ef6c00',   # orange
    'Struggler':            '#c62828',   # red
}

ENROLMENT_COHORTS = ['2025-Q1', '2025-Q2', '2025-Q3', '2025-Q4']

# How strongly a single study-week advances a concept (multiple sessions/week).
LEARN_INTENSITY = 1.55


@dataclass
class StudentProfile:
    """A single learner: demographics + evolving knowledge state."""
    student_id:                 str
    prior_knowledge_vector:     np.ndarray      # 15-dim, initial mastery
    mastery_vector:             np.ndarray      # 15-dim, evolving mastery
    learning_preference:        str             # visual / reading / practice
    available_study_hours:      float           # hours per day
    learning_goal:              str             # speed / depth / certification
    learning_rate:              float           # individual ability coefficient
    # ── realistic demographic / context attributes ──────────────
    archetype:                  str   = 'Steady Learner'
    age:                        int   = 25
    education_background:        str   = 'Undergraduate'
    weekly_active_days:         int   = 4
    enrolment_cohort:           str   = '2025-Q1'
    # ── outcome fields (filled after simulation) ────────────────
    satisfaction_score:         float = 0.5
    completion_rate:            float = 0.40
    final_concepts_mastered:    int   = 0
    total_study_hours:          float = 0.0
    episode_history:            List[Dict[str, Any]] = field(default_factory=list)

    def get_goal_encoding(self) -> int:
        return LEARNING_GOALS.index(self.learning_goal)

    def get_mastery_bins(self) -> Tuple[int, ...]:
        """Discretise mastery vector into bins for state encoding."""
        bins = CONFIG['mastery_bins']
        return tuple(int(np.digitize(m, bins) - 1) for m in self.mastery_vector)

    def get_hours_bin(self) -> int:
        return int(np.digitize(self.available_study_hours, CONFIG['hours_bins']) - 1)


class DataGenerator:
    """
    Module 1 — builds a realistic 200-learner cohort and simulates a
    coherent, curriculum-ordered learning journey for each student.

    Mastery grows per study-week as:
        gain  = lr · (1−m) · affinity · difficulty · prereq · hours · INTENSITY
        m_new = m_old + gain
    Students always study the lowest unmet, prerequisite-satisfied
    concept (topological order), producing clean S-shaped learning curves.
    """

    def __init__(self, rng: np.random.Generator):
        self.rng = rng

    # ── public API ────────────────────────────────────────────────

    def _assign_archetypes(self, n: int) -> List[str]:
        names  = list(LEARNER_ARCHETYPES.keys())
        shares = np.array([LEARNER_ARCHETYPES[a]['share'] for a in names])
        shares = shares / shares.sum()
        counts = np.floor(shares * n).astype(int)
        while counts.sum() < n:                      # distribute rounding remainder
            counts[int(np.argmax(shares))] += 1
        labels: List[str] = []
        for name, c in zip(names, counts):
            labels.extend([name] * int(c))
        self.rng.shuffle(labels)
        return labels[:n]

    def generate_students(self, n: int = CONFIG['n_students']) -> List[StudentProfile]:
        students:   List[StudentProfile] = []
        archetypes = self._assign_archetypes(n)

        for i, arch in enumerate(archetypes):
            spec  = LEARNER_ARCHETYPES[arch]
            sid   = f"S{i+1:03d}"

            lr    = float(np.clip(self.rng.normal(*spec['lr']), 0.05, 0.40))
            hours = float(round(self.rng.uniform(*spec['hours']), 1))
            pref  = str(self.rng.choice(LEARNING_PREFERENCES))
            goal  = str(self.rng.choice(spec['goals']))
            age   = int(self.rng.integers(spec['age'][0], spec['age'][1] + 1))
            edu   = str(self.rng.choice(spec['edu']))
            days  = int(self.rng.integers(spec['days'][0], spec['days'][1] + 1))
            cohort = str(self.rng.choice(ENROLMENT_COHORTS))

            # Prior knowledge is concentrated on the foundational concepts.
            prior  = np.zeros(N_CONCEPTS)
            n_prior = int(self.rng.integers(spec['prior_n'][0], spec['prior_n'][1] + 1))
            for j in range(min(n_prior, 5)):
                prior[j] = float(round(self.rng.uniform(*spec['prior_lvl']), 2))

            students.append(StudentProfile(
                student_id=sid,
                prior_knowledge_vector=prior.copy(),
                mastery_vector=prior.copy(),
                learning_preference=pref,
                available_study_hours=hours,
                learning_goal=goal,
                learning_rate=lr,
                archetype=arch,
                age=age,
                education_background=edu,
                weekly_active_days=days,
                enrolment_cohort=cohort,
            ))
        return students

    # ── one study-week on a single concept ───────────────────────

    def simulate_learning_episode(self,
                                   student: StudentProfile,
                                   concept_id: str,
                                   kg: 'KnowledgeGraph') -> Dict[str, Any]:
        """Simulate one study-week. Returns the updated mastery + metrics."""
        idx   = CONCEPT_INDEX[concept_id]
        old_m = float(student.mastery_vector[idx])

        ctype    = CONCEPTS[concept_id]['ctype']
        affinity = PREFERENCE_AFFINITY[student.learning_preference][ctype]
        diff_factor  = 1.0 - (CONCEPTS[concept_id]['difficulty'] - 1) * 0.10
        prereqs_met  = kg.prerequisites_satisfied(concept_id, student.mastery_vector)
        prereq_mult  = 1.0 if prereqs_met else 0.30
        # Study hours materially affect the speed of learning.
        hours_factor = float(np.clip(student.available_study_hours / 4.0, 0.45, 1.70))

        gain  = (student.learning_rate * (1 - old_m) * affinity
                 * diff_factor * prereq_mult * hours_factor * LEARN_INTENSITY)
        new_m = float(np.clip(old_m + gain, 0.0, 1.0))
        student.mastery_vector[idx] = new_m

        # Quiz score: observed performance = mastery + measurement noise.
        quiz_score = float(np.clip(
            new_m + self.rng.normal(0, CONFIG['quiz_noise_std']), 0, 1))

        # Completion rate grows with the number of concepts mastered.
        mastered_count = int(np.sum(student.mastery_vector >= CONFIG['mastery_threshold']))
        student.completion_rate = float(np.clip(
            CONFIG['init_completion_rate'] + 0.04 * mastered_count, 0, 1))

        # Satisfaction = progress felt + content alignment − frustration.
        progress_term = float(np.clip(gain * 2.6, 0.0, 0.42))
        affinity_term = (affinity - 0.55) * 0.45
        prereq_term   = 0.0 if prereqs_met else -0.28
        goal_term     = (GOAL_IMPORTANCE[student.learning_goal][CONCEPTS[concept_id]['difficulty']]
                         - 0.6) * 0.20
        noise         = float(self.rng.normal(0, 0.03))
        satisfaction  = float(np.clip(
            0.50 + progress_term + affinity_term + prereq_term + goal_term + noise,
            0.05, 0.99))
        student.satisfaction_score = satisfaction

        record: Dict[str, Any] = {
            'student_id':         student.student_id,
            'archetype':          student.archetype,
            'concept_id':         concept_id,
            'concept_name':       CONCEPTS[concept_id]['name'],
            'difficulty':         CONCEPTS[concept_id]['difficulty'],
            'old_mastery':        old_m,
            'new_mastery':        new_m,
            'delta_mastery':      new_m - old_m,
            'quiz_score':         quiz_score,
            'study_hours':        float(CONCEPTS[concept_id]['est_hours']),
            'completion_rate':    student.completion_rate,
            'satisfaction':       satisfaction,
            'prereqs_met':        prereqs_met,
        }
        student.episode_history.append(record)
        return record

    # ── pick the next concept on the curriculum path ─────────────

    def _select_target(self, student: StudentProfile, kg: 'KnowledgeGraph') -> str:
        """Lowest-difficulty, prerequisite-satisfied, not-yet-mastered concept."""
        thr   = CONFIG['mastery_threshold']
        valid = kg.get_valid_actions(student.mastery_vector)
        pool  = [c for c in valid if student.mastery_vector[CONCEPT_INDEX[c]] < thr]
        if not pool:
            pool = valid
        pool.sort(key=lambda c: (CONCEPTS[c]['difficulty'], CONCEPT_INDEX[c]))
        # A little exploration: occasionally take the 2nd-easiest option.
        if len(pool) > 1 and self.rng.random() < 0.18:
            return pool[1]
        return pool[0]

    def build_dataset(self,
                      students: List[StudentProfile],
                      kg: 'KnowledgeGraph') -> pd.DataFrame:
        """Simulate the full curriculum journey for every student."""
        rows: List[Dict[str, Any]] = []
        for s in students:
            for ep in range(CONFIG['n_learning_episodes']):
                cid = self._select_target(s, kg)
                rec = self.simulate_learning_episode(s, cid, kg)
                rec['episode']             = ep
                rec['learning_preference'] = s.learning_preference
                rec['learning_goal']       = s.learning_goal
                rec['available_hours']     = s.available_study_hours
                rows.append(rec)
            # Freeze final outcome fields on the profile.
            s.final_concepts_mastered = int(np.sum(s.mastery_vector >= CONFIG['mastery_threshold']))
            s.total_study_hours       = float(s.available_study_hours
                                              * CONFIG['n_learning_episodes'])
        return pd.DataFrame(rows)

    # ── tidy one-row-per-student table (the "real" dataset view) ──

    def build_student_table(self, students: List[StudentProfile]) -> pd.DataFrame:
        thr = CONFIG['mastery_threshold']
        rows = [{
            'Student':        s.student_id,
            'Archetype':      s.archetype,
            'Age':            s.age,
            'Education':      s.education_background,
            'Preference':     s.learning_preference,
            'Goal':           s.learning_goal,
            'Hours/Day':      round(s.available_study_hours, 1),
            'Active Days/Wk': s.weekly_active_days,
            'Learn Rate':     round(s.learning_rate, 3),
            'Cohort':         s.enrolment_cohort,
            'Prior Known':    int(np.sum(s.prior_knowledge_vector >= 0.40)),
            'Mastered/15':    int(np.sum(s.mastery_vector >= thr)),
            'Avg Mastery':    round(float(np.mean(s.mastery_vector)), 2),
            'Satisfaction':   round(s.satisfaction_score, 2),
            'Completion %':   int(round(s.completion_rate * 100)),
        } for s in students]
        return pd.DataFrame(rows)


print("✅ DataGenerator and StudentProfile defined (4 realistic archetypes)")


✅ DataGenerator and StudentProfile defined (4 realistic archetypes)
